# 🛡️ AEGIS — Selective Multimodal Reasoning for Privacy-Preserving Edge Surveillance
### A cheap temporal triage model watches every frame. **Local multimodal Gemma 4** wakes up only for suspicious moments, independently inspects anonymized visual evidence, and vetoes false alarms *before* dispatch.

**Build with Gemma — Track 2: AI for Public Safety · Kaggle · 2026**

> **One-sentence pitch:** AEGIS turns a noisy CCTV anomaly detector into a trustworthy incident system by making an on-device **Gemma 4** the skeptical forensic verifier that must be convinced by actual (anonymized) visual evidence before anything is escalated.

## 1 · The Problem

Conventional surveillance analytics fail operators in three compounding ways:

1. **Alarm fatigue.** Per-frame anomaly thresholds fire on headlights, occlusions, and compression bursts. Operators learn to ignore alarms — which is worse than having none.
2. **Cloud latency & privacy risk.** Streaming raw video to a cloud VLM adds seconds of latency and ships citizens' faces off-site. Both are unacceptable for public-safety infrastructure.
3. **Black-box scores.** A number like `P(anomaly)=0.87` cannot be audited, cross-examined, or acted on with confidence. Dispatchers need *evidence*, not scores.

## 2 · Our Solution — and Why Gemma 4 Is Essential

**SafetyFormer** (a compact temporal transformer over frozen visual embeddings) performs *continuous, millisecond-cost triage* on every temporal window. It is deliberately tuned for **high recall**: it is allowed to be paranoid, because it does not raise alarms — it nominates **candidate incidents**.

Every candidate then wakes **Gemma 4 running locally on the edge device**. Gemma receives 4–8 **anonymized evidence keyframes** from the most suspicious temporal region plus the detector's telemetry — explicitly labeled an *untrusted hypothesis* — and acts as a **skeptical forensic verifier**: it must find visually supported evidence of a genuine incident, or the candidate is suppressed/downgraded.

**Why a numeric detector cannot do this alone:** an anomaly score cannot distinguish *fighting* from *a crowd celebrating*, or *a robbery* from *a shopkeeper restocking at night*. That distinction requires open-world visual understanding — reading scene context, actors, objects, and interactions. That is precisely what a multimodal LLM provides and what no calibrated score can. **Remove Gemma 4 and the system collapses back into an uninspectable score threshold** — Gemma is load-bearing, not decorative.

The selective-compute design gives a three-way win:

| Advantage | Mechanism |
|---|---|
| **Compute efficiency** | Gemma runs only on gate-passing candidates — we measure the *Gemma activation rate* and VLM calls avoided below |
| **Privacy** | Faces are pixelated on-device **before** Gemma sees any frame; Gemma weights run locally; raw video never leaves the edge |
| **False-alarm reduction** | A verifier that *looks at the evidence* rejects score-only false positives — measured in the ablation |

## 3 · System Architecture

```
┌─────────────────────────────── EDGE DEVICE (everything below is local) ───────────────────────────────┐
│                                                                                                        │
│  CCTV frames ──► FACE ANONYMIZATION ──► frozen visual encoder ──► per-frame embeddings                 │
│                  (privacy firewall)                                     │                              │
│                                                                         ▼                              │
│                                       SAFETYFORMER temporal triage (dense 16-frame windows,            │
│                                       top-k MIL trained, calibrated) ──► window anomaly scores         │
│                                                                         │                              │
│                              uncertainty + persistence + hysteresis GATES (false-alarm suppression)    │
│                                                                         │ candidate incident only      │
│                                                                         ▼                              │
│               EVIDENCE KEYFRAME SELECTION (top anomaly region, temporal-diversity sampling)            │
│                                                                         │ 4–8 anonymized frames        │
│                                                                         ▼                              │
│               ★ LOCAL MULTIMODAL GEMMA 4 FORENSIC VERIFIER ★ (4-bit, no cloud API)                    │
│                 skeptical visual inspection → structured JSON verdict                                  │
│                                                                         │                              │
│                              INCIDENT DECISION ENGINE (detector × Gemma fusion)                        │
│                                       SUPPRESS / REVIEW / ESCALATE                                     │
│                                                                         │                              │
│               structured evidence-backed incident report ──► AES-256-GCM SECURE DISPATCH CHANNEL ──►   │
└────────────────────────────────────────────────────────────────────────────────────────────────────────┘
```

**Dataset note:** UCF-Crime is used here as a *prototype benchmark* for real-world surveillance anomalies. Real deployment requires domain-specific validation, and we say so explicitly in the Limitations section.

## 4 · Setup & Reproducibility

Installs are pinned to what the pipeline actually needs. Gemma 4 requires a recent `transformers` (v5+); `bitsandbytes` enables 4-bit NF4 quantization so the multimodal verifier fits comfortably in a 16 GB T4 — the same memory class as edge AI boxes (Jetson Orin, etc.).

In [1]:
import importlib, subprocess, sys, warnings
warnings.filterwarnings("ignore")

def ensure(pip_spec, import_name):
    """Import a package, pip-installing/upgrading it first if necessary."""
    try:
        importlib.import_module(import_name)
        print(f"  [ok]        {pip_spec}")
        return True
    except ImportError:
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                                   "--no-warn-conflicts", "-U", pip_spec], timeout=600)
            importlib.import_module(import_name)
            print(f"  [installed] {pip_spec}")
            return True
        except Exception as e:
            print(f"  [FAILED]    {pip_spec}: {type(e).__name__}")
            return False

print("Dependencies:")
# Gemma 4 support landed in transformers v5 — upgrade unconditionally.
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-warn-conflicts",
                       "-U", "transformers>=5.0", "accelerate"], timeout=900)
import transformers
print(f"  [ok]        transformers {transformers.__version__}")
HAS_BNB    = ensure("bitsandbytes", "bitsandbytes")   # 4-bit quantization (GPU only)
HAS_TIMM   = ensure("timm", "timm")                   # frozen visual backbones
HAS_CRYPTO = ensure("cryptography", "cryptography")   # AES-256-GCM dispatch channel
print("Setup complete.")

Dependencies:
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 50.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 22.7 MB/s eta 0:00:00
  [ok]        transformers 5.13.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 27.0 MB/s eta 0:00:00
  [installed] bitsandbytes
  [ok]        timm
  [ok]        cryptography
Setup complete.


In [ ]:
import os, io, re, json, math, time, random, hashlib
from dataclasses import dataclass, asdict, field
from pathlib import Path
from collections import defaultdict, deque

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_auc_score, average_precision_score, roc_curve,
                             precision_recall_curve, confusion_matrix, f1_score,
                             precision_score, recall_score, brier_score_loss)
from tqdm.auto import tqdm
from IPython.display import Markdown, display

def seed_everything(seed: int):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"torch {torch.__version__} | device: {DEVICE}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {p.name} | {p.total_memory / 1e9:.1f} GB")

# Global results registry — the final summary reports ONLY values written here at runtime.
RESULTS = {}

## 5 · Configuration

One dataclass holds every knob. **`DEBUG=True` runs a small subset for development; the submitted notebook runs with `DEBUG=False`** (full training/validation/test data). A loud banner makes the active mode unmissable.

In [ ]:
@dataclass
class Config:
    seed: int = 42
    DEBUG: bool = False               # True = tiny dev subset. FINAL RUN MUST BE False.

    # ---- data discovery (searched in order; adapts to the attached dataset)
    data_roots: tuple = ("/kaggle/input", "./input", "./data", ".")
    image_exts: tuple = (".png", ".jpg", ".jpeg", ".bmp", ".webp")
    video_exts: tuple = (".mp4", ".avi", ".mov", ".mkv", ".webm", ".mpg", ".mpeg")
    normal_keywords: tuple = ("normal", "safe", "benign", "neutral", "no_event", "negative")
    val_frac: float = 0.15            # carved from train, video-level, stratified
    test_frac: float = 0.15           # only if the dataset ships without a test split
    max_frames_per_video: int = 1200  # uniform decimation cap for very long videos

    # ---- dense temporal windows (MIL instances)
    window_len: int = 16              # frames per window
    window_stride: int = 8            # 50% overlap -> dense coverage

    # ---- frozen visual backbones (A/B compared on validation ROC-AUC)
    backbone_candidates: tuple = (
        ("clip_vitb32",     "vit_base_patch32_clip_224.openai"),
        ("convnextv2_tiny", "convnextv2_tiny.fcmae_ft_in22k_in1k"),
    )
    ab_train_frac: float = 0.25       # fraction of train videos used for the backbone A/B
    extract_batch: int = 256

    # ---- SafetyFormer (compact temporal transformer)
    d_model: int = 256
    n_heads: int = 4
    n_layers: int = 2
    dropout: float = 0.20             # doubles as MC-dropout at inference
    mc_passes: int = 12

    # ---- top-k MIL + training
    mil_alpha_grid: tuple = (0.05, 0.10, 0.20, 0.30)   # tuned on VALIDATION only
    windows_per_video_train: int = 64
    batch_videos: int = 16
    epochs: int = 25
    lr: float = 3e-4
    weight_decay: float = 0.05
    warmup_frac: float = 0.10
    grad_clip: float = 1.0
    patience: int = 5                 # early stopping on validation ROC-AUC
    subtype_loss_weight: float = 0.3  # auxiliary head, gradient-detached from trunk

    # ---- alarm gates (false-alarm suppression)
    sup_ema_alpha: float = 0.40
    sup_window: int = 7
    sup_k_required: int = 4
    sup_std_max: float = 0.15         # MC-dropout std rejection gate
    sup_off_ratio: float = 0.60       # hysteresis: clear below on_threshold * ratio
    val_min_recall: float = 0.85      # threshold rule: recall-first triage (see §12)

    # ---- Gemma 4 forensic verifier (LOCAL inference)
    gemma_candidates: tuple = ("google/gemma-4-E4B-it", "google/gemma-4-E2B-it")
    gemma_max_new_tokens: int = 512   # headroom for Gemma 4 thinking tokens before the JSON
    n_evidence_frames: int = 6
    evidence_size: int = 432          # Gemma 4 accepts variable sizes (divisible-by-48 friendly)
    ablation_videos_per_class: int = 20   # labeled Gemma-evaluation subset size (per class)
    demo_cases: int = 2               # case studies per outcome type

    # ---- artifacts
    cache_dir: str = "/kaggle/working/cache" if os.path.isdir("/kaggle/working") else "./cache"
    out_dir: str = "/kaggle/working" if os.path.isdir("/kaggle/working") else "./outputs"

    def __post_init__(self):
        if self.DEBUG:
            self.epochs = 6
            self.ab_train_frac = 0.10
            self.ablation_videos_per_class = 4
        Path(self.cache_dir).mkdir(parents=True, exist_ok=True)
        Path(self.out_dir).mkdir(parents=True, exist_ok=True)

CFG = Config()
seed_everything(CFG.seed)

if CFG.DEBUG:
    print("!" * 78)
    print("!!  DEBUG MODE — TINY SUBSET, NOT A VALID EVALUATION. Set DEBUG=False for  !!")
    print("!!  the final run. All reported metrics below would be meaningless.        !!")
    print("!" * 78)
else:
    print("FULL RUN: complete train/validation/test data will be used.")
print("-" * 60)
for k, v in asdict(CFG).items():
    print(f"  {k:26s} = {v}")

## 6 · Dataset & Leakage-Free Video-Level Splits

**UCF-Crime facts that shape everything downstream:** labels are **video-level** (an "Assault" video is mostly ordinary footage with a short anomalous segment), and frames from one video are near-duplicates. Two consequences:

1. **Splitting must happen at the video level** — every window of a video lives in exactly one split. We assert this programmatically below.
2. **Training must not assume every window of an anomalous video is anomalous** — that is weak-label poisoning. Section 10 fixes it with top-k Multiple Instance Learning.

Discovery adapts to the attached dataset layout: split folders (`Train/Test`) at any depth are honored, classes come from the owning folder, and frame files are grouped back into their source videos via the trailing-index pattern (`Abuse001_x264_930.png → Abuse001_x264`).

In [ ]:
SPLIT_ALIASES = {"train": "train", "training": "train", "test": "test", "testing": "test",
                 "val": "val", "valid": "val", "validation": "val", "dev": "val"}
FRAME_SUFFIX_RE = re.compile(r"^(.+?)[_\-. ]?(\d+)$")

def _infer_split(rel_parts):
    for p in rel_parts:
        alias = SPLIT_ALIASES.get(p.strip().lower())
        if alias:
            return alias
    return None

def _find_leaf_media_dirs(root: Path, cfg):
    exts = set(cfg.image_exts) | set(cfg.video_exts)
    leaves = []
    for dirpath, _dirs, files in os.walk(root):
        media = [f for f in files if os.path.splitext(f)[1].lower() in exts]
        if media:
            leaves.append((Path(dirpath), media))
    return leaves

def _classify_leaf(leaf: Path, root: Path):
    parent = leaf.parent
    if leaf == root or leaf.name.strip().lower() in SPLIT_ALIASES:
        return "uncategorized", None
    if parent == root or parent.name.strip().lower() in SPLIT_ALIASES or leaf == root:
        return leaf.name, None                    # root/[Train/]Class/*.ext
    return parent.name, leaf.name                 # root/[Train/]Class/video01/*.ext

def _group_frames(files, video_hint):
    groups = defaultdict(list)
    for f in files:
        stem = os.path.splitext(f)[0]
        m = FRAME_SUFFIX_RE.match(stem)
        if video_hint is not None:
            key, order = video_hint, int(m.group(2)) if m else 0
        elif m:
            key, order = m.group(1), int(m.group(2))
        else:
            key, order = stem, 0
        groups[key].append((order, f))
    return {k: [f for _, f in sorted(v)] for k, v in groups.items()}

def discover_dataset(cfg) -> pd.DataFrame:
    """Scan cfg.data_roots; return one row per source video (ordered frame paths)."""
    records, scanned = [], None
    for root_str in cfg.data_roots:
        root = Path(root_str)
        if not root.is_dir():
            continue
        sub_roots = ([d for d in root.iterdir() if d.is_dir()]
                     if root_str.endswith("input") else [root]) or [root]
        for ds_root in sub_roots:
            for leaf, files in _find_leaf_media_dirs(ds_root, cfg):
                rel = leaf.relative_to(ds_root).parts
                split = _infer_split(rel)
                cls, hint = _classify_leaf(leaf, ds_root)
                imgs = [f for f in files if os.path.splitext(f)[1].lower() in cfg.image_exts]
                for grp, frames in _group_frames(imgs, hint).items():
                    records.append(dict(class_name=cls, split=split, group=f"{cls}::{grp}",
                                        paths=[str(leaf / f) for f in frames]))
            if records:
                scanned = str(ds_root)
        if records:
            break
    if not records:
        raise FileNotFoundError(f"No media under {cfg.data_roots}. Attach the UCF-Crime "
                                "frame dataset and re-run.")
    df = pd.DataFrame(records)
    df["n_frames"] = df.paths.str.len()
    df = df[df.n_frames >= 4].reset_index(drop=True)
    dup = df.groupby("group").cumcount()
    df.loc[dup > 0, "group"] = df.loc[dup > 0, "group"] + "__d" + dup[dup > 0].astype(str)
    df["is_anomalous"] = ~df.class_name.str.lower().str.contains("|".join(CFG.normal_keywords))
    print(f"Scanned root: {scanned}")
    return df

def decimate_paths(paths, cap):
    """Uniform decimation for very long videos (keeps whole-video coverage)."""
    if len(paths) <= cap:
        return list(paths)
    idx = np.linspace(0, len(paths) - 1, cap).round().astype(int)
    return [paths[i] for i in np.unique(idx)]

def load_frame(path, size=None):
    img = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if img is None:
        return None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    if size is not None:
        img = cv2.resize(img, (size, size), interpolation=cv2.INTER_CUBIC)
    return img

manifest = discover_dataset(CFG)
manifest["feat_paths"] = manifest.paths.apply(lambda p: decimate_paths(p, CFG.max_frames_per_video))
manifest["n_feat_frames"] = manifest.feat_paths.str.len()

CLASS_NAMES = sorted(manifest.class_name.unique())
ANOMALY_CLASSES = sorted(manifest[manifest.is_anomalous].class_name.unique())
SUBTYPE_TO_IDX = {c: i for i, c in enumerate(ANOMALY_CLASSES)}
manifest["subtype_idx"] = manifest.class_name.map(SUBTYPE_TO_IDX).fillna(-1).astype(int)

print(f"Videos: {len(manifest)} | classes: {len(CLASS_NAMES)} "
      f"({len(ANOMALY_CLASSES)} anomaly subtypes) | frames: {manifest.n_frames.sum():,} "
      f"(after decimation cap: {manifest.n_feat_frames.sum():,})")
print(f"Anomalous videos: {int(manifest.is_anomalous.sum())} | "
      f"normal videos: {int((~manifest.is_anomalous).sum())}")

# a frame is ~1/10th-subsampled 30fps video in the standard UCF-Crime frame release ->
# one 16-frame window spans roughly 5 s of wall-clock time. We verify frame indices are
# ordered and report the median inter-frame index gap actually present in the data.
_gaps = []
for p in manifest.paths.iloc[:50]:
    ids = [int(FRAME_SUFFIX_RE.match(os.path.splitext(os.path.basename(x))[0]).group(2))
           for x in p[:200] if FRAME_SUFFIX_RE.match(os.path.splitext(os.path.basename(x))[0])]
    if len(ids) > 2:
        _gaps.extend(np.diff(sorted(ids)).tolist())
if _gaps:
    print(f"Median source-frame index gap between consecutive stored frames: {int(np.median(_gaps))} "
          f"-> a {CFG.window_len}-frame window is temporally coherent local context, "
          "not scattered snapshots.")

In [ ]:
def assign_splits(df, cfg) -> pd.DataFrame:
    """VIDEO-LEVEL splits. Honors dataset-provided train/test; carves stratified val
    from train. Every window of a video inherits the video's split — no exceptions."""
    df = df.copy()
    if not df.split.isin(["test"]).any():          # dataset shipped without a test split
        tr_idx, te_idx = train_test_split(df.index, test_size=cfg.test_frac,
                                          stratify=df.class_name, random_state=cfg.seed)
        df.loc[te_idx, "split"] = "test"
        df.loc[tr_idx, "split"] = "train"
    df.loc[~df.split.isin(["train", "test", "val"]), "split"] = "train"
    if not (df.split == "val").any():
        tr = df[df.split == "train"]
        try:
            _, va_idx = train_test_split(tr.index, test_size=cfg.val_frac,
                                         stratify=tr.class_name, random_state=cfg.seed)
        except ValueError:
            _, va_idx = train_test_split(tr.index, test_size=cfg.val_frac,
                                         random_state=cfg.seed)
        df.loc[va_idx, "split"] = "val"
    return df

manifest = assign_splits(manifest, CFG)

if CFG.DEBUG:   # dev-only subset; the final notebook runs with DEBUG=False
    caps = {"train": 24, "val": 8, "test": 8}
    manifest = (manifest.groupby(["split", "class_name"], group_keys=False)
                .apply(lambda g: g.sample(min(len(g), caps[g.name[0]]), random_state=CFG.seed))
                .reset_index(drop=True))
    print("DEBUG subset in effect — NOT a valid evaluation.")

# ---- leakage-free guarantee: video IDs are disjoint across splits (asserted, visibly)
train_ids = set(manifest[manifest.split == "train"].group)
val_ids   = set(manifest[manifest.split == "val"].group)
test_ids  = set(manifest[manifest.split == "test"].group)
assert train_ids.isdisjoint(val_ids),  "LEAKAGE: train/val share videos!"
assert train_ids.isdisjoint(test_ids), "LEAKAGE: train/test share videos!"
assert val_ids.isdisjoint(test_ids),   "LEAKAGE: val/test share videos!"
print("Split-overlap assertions PASSED: no video appears in more than one split.\n")

summary = (manifest.assign(label=np.where(manifest.is_anomalous, "anomalous", "normal"))
           .groupby(["split", "label"]).size().unstack(fill_value=0))
summary["total"] = summary.sum(1)
print(summary.loc[["train", "val", "test"]].to_string())
RESULTS["split_counts"] = summary.to_dict()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
cls_counts = manifest.groupby("class_name").size().sort_values()
colors = ["#21918c" if not manifest[manifest.class_name == c].is_anomalous.iloc[0]
          else "#e45756" for c in cls_counts.index]
axes[0].barh(cls_counts.index, cls_counts.values, color=colors)
axes[0].set_title("Videos per class (red = anomalous, teal = normal)")
axes[0].set_xlabel("videos")
axes[1].hist(manifest.n_feat_frames, bins=40, color="#3b528b")
axes[1].set_title("Frames per video (after decimation cap)")
axes[1].set_xlabel("frames"); axes[1].set_ylabel("videos")
plt.tight_layout(); plt.show()

## 7 · Privacy-First Frame Processing

**Contract:** identity is removed *before* any model — including Gemma 4 — sees a frame, and every model runs locally.

* **Face pixelation** (Haar cascade, irreversible mosaic) runs on-device before evidence frames are assembled. *Honest scope:* at UCF-Crime's low source resolution faces are rarely resolvable to begin with, and Haar detection is best-effort — we anonymize what is detectable and state that this is not a formal de-identification guarantee. Bodies, clothing, and context remain visible by design (responders need them).
* **AES-256-GCM secure dispatch** — only the *final* structured incident report plus selected encrypted evidence frames are prepared for transmission to a dispatch hub. GCM gives confidentiality **and** tamper detection; a fresh 96-bit nonce per message prevents replay.
* **What never leaves the device:** raw video, continuous frames, un-anonymized imagery, and Gemma prompts/outputs (inference is local — verified in §13).

In [ ]:
from cryptography.hazmat.primitives.ciphers.aead import AESGCM

class FaceAnonymizer:
    """Best-effort on-device face pixelation (irreversible mosaic)."""
    def __init__(self, pixel_blocks: int = 6):
        self.face_detector, self.pixel_blocks = None, pixel_blocks
        try:
            cascade = os.path.join(cv2.data.haarcascades, "haarcascade_frontalface_default.xml")
            self.face_detector = cv2.CascadeClassifier(cascade)
        except AttributeError:
            warnings.warn("cv2 Haar cascades unavailable — face pixelation disabled")

    def _pixelate(self, roi):
        h, w = roi.shape[:2]
        small = cv2.resize(roi, (max(1, self.pixel_blocks), max(1, self.pixel_blocks)))
        return cv2.resize(small, (w, h), interpolation=cv2.INTER_NEAREST)

    def anonymize(self, frame_rgb):
        if self.face_detector is None:
            return frame_rgb.copy(), 0
        gray = cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2GRAY)
        faces = self.face_detector.detectMultiScale(gray, 1.1, 4, minSize=(14, 14))
        out = frame_rgb.copy()
        for (x, y, w, h) in faces:
            out[y:y + h, x:x + w] = self._pixelate(out[y:y + h, x:x + w])
        return out, len(faces)

    def anonymize_clip(self, clip):
        results = [self.anonymize(f) for f in clip]
        return np.stack([r[0] for r in results]), sum(r[1] for r in results)

class SecureChannel:
    """AES-256-GCM authenticated encryption for the dispatch payload (edge -> hub)."""
    def __init__(self, key: bytes = None):
        self.key = key or AESGCM.generate_key(bit_length=256)
        self._aes = AESGCM(self.key)

    @staticmethod
    def _serialize(payload: dict) -> bytes:
        buf = io.BytesIO()
        arrays = {k: v for k, v in payload.items() if isinstance(v, np.ndarray)}
        meta = {k: v for k, v in payload.items() if not isinstance(v, np.ndarray)}
        np.savez_compressed(buf, __meta__=np.frombuffer(
            json.dumps(meta, default=str).encode(), dtype=np.uint8), **arrays)
        return buf.getvalue()

    @staticmethod
    def _deserialize(raw: bytes) -> dict:
        with np.load(io.BytesIO(raw), allow_pickle=False) as z:
            out = {k: z[k] for k in z.files if k != "__meta__"}
            out.update(json.loads(z["__meta__"].tobytes().decode()))
        return out

    def encrypt(self, payload: dict) -> bytes:
        nonce = os.urandom(12)
        return nonce + self._aes.encrypt(nonce, self._serialize(payload), b"aegis-v2")

    def decrypt(self, blob: bytes) -> dict:
        return self._deserialize(self._aes.decrypt(blob[:12], blob[12:], b"aegis-v2"))

anonymizer = FaceAnonymizer()
channel = SecureChannel()

# ---- visual: ORIGINAL -> ANONYMIZED (upscaled for visibility; source is low-res CCTV)
_rec = manifest[manifest.split == "test"].iloc[0]
_raw = load_frame(_rec.feat_paths[len(_rec.feat_paths) // 2], size=CFG.evidence_size)
_anon, _n = anonymizer.anonymize(_raw)
fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))
axes[0].imshow(_raw);  axes[0].set_title("ORIGINAL FRAME (never leaves device)")
axes[1].imshow(_anon); axes[1].set_title(f"ANONYMIZED ({_n} face region(s) pixelated)")
for ax in axes: ax.set_axis_off()
plt.suptitle(f"Privacy firewall — {_rec.group}", y=1.02)
plt.tight_layout(); plt.show()

# ---- encrypted round-trip + tamper detection
_blob = channel.encrypt({"report": "demo", "evidence_jpg": np.frombuffer(
    cv2.imencode(".jpg", cv2.cvtColor(_anon, cv2.COLOR_RGB2BGR))[1].tobytes(), dtype=np.uint8)})
_back = channel.decrypt(_blob)
print(f"Dispatch payload round-trip OK ({len(_blob)/1024:.1f} KB encrypted).")
try:
    channel.decrypt(_blob[:-1] + bytes([_blob[-1] ^ 0xFF]))
except Exception as e:
    print(f"Tampered payload REJECTED ({type(e).__name__}) — GCM authentication works.")

## 8 · Frozen Visual Feature Extraction (cached once, reused everywhere)

Frames are embedded **once** by a frozen encoder and cached to disk per video — training then iterates over embeddings, never pixels, which is what makes the experiment table in §11 affordable.

Because the standard UCF-Crime frame release is **low-resolution (64×64)**, backbone choice is an empirical question, not a doctrine: fine spatial detail is gone, so we A/B a **semantic** encoder (CLIP ViT-B/32 — CLIP-family features dominate the recent UCF-Crime literature) against the **texture-strong** ConvNeXt-V2 on the *same validation split* in §9, and let validation ROC-AUC decide.

In [ ]:
import timm

def build_backbone(model_name):
    model = timm.create_model(model_name, pretrained=True, num_classes=0)
    data_cfg = timm.data.resolve_model_data_config(model)
    spec = dict(name=model_name, size=data_cfg["input_size"][-1],
                mean=data_cfg["mean"], std=data_cfg["std"])
    model = model.eval().to(DEVICE)
    for p in model.parameters():
        p.requires_grad_(False)
    return model, model.num_features, spec

def preprocess_frames(frames, spec):
    """frames: list/array of RGB uint8 -> float tensor [N,3,S,S], backbone-normalized."""
    S = spec["size"]
    mean = torch.tensor(spec["mean"]).view(1, 3, 1, 1)
    std = torch.tensor(spec["std"]).view(1, 3, 1, 1)
    x = torch.from_numpy(np.stack(
        [cv2.resize(f, (S, S), interpolation=cv2.INTER_CUBIC) for f in frames]))
    x = x.permute(0, 3, 1, 2).float().div_(255.0)
    return (x - mean) / std

def _cache_path(tag, group):
    d = Path(CFG.cache_dir) / f"feats_{tag}"
    d.mkdir(parents=True, exist_ok=True)
    return d / (hashlib.md5(group.encode()).hexdigest() + ".npy")

@torch.no_grad()
def embed_frames_now(frames, backbone, spec):
    """Embed a list of RGB frames on the GPU (used by both extraction and the live demo)."""
    out = []
    for s in range(0, len(frames), CFG.extract_batch):
        x = preprocess_frames(frames[s:s + CFG.extract_batch], spec).to(DEVICE)
        with torch.autocast("cuda", enabled=DEVICE.type == "cuda"):
            out.append(backbone(x).half().cpu())
    return torch.cat(out).numpy()

def extract_features(records, tag, backbone, spec):
    """Per-video .npy cache -> dict {group: float16 [T, D]}. Incremental & resumable."""
    feats, todo = {}, []
    for r in records:
        p = _cache_path(tag, r["group"])
        if p.exists():
            feats[r["group"]] = np.load(p)
        else:
            todo.append(r)
    for r in tqdm(todo, desc=f"Embedding [{tag}]", disable=not todo):
        frames = [f for f in (load_frame(pp) for pp in r["feat_paths"]) if f is not None]
        if len(frames) < 4:
            continue
        emb = embed_frames_now(frames, backbone, spec)
        np.save(_cache_path(tag, r["group"]), emb)
        feats[r["group"]] = emb
    return feats

print("Feature extraction utilities ready (per-video cache under "
      f"{CFG.cache_dir}/feats_<backbone>/).")

## 9 · SafetyFormer — Compact Temporal Triage Model

SafetyFormer scores **one dense 16-frame window at a time**: frozen embeddings → projection → learned positional encoding → a 2-layer pre-norm Transformer encoder → **learned-query attention pooling** (its attention weights are a free per-frame saliency signal) → a single anomaly logit.

It is deliberately small (~1M trainable parameters): capacity goes into local temporal dynamics, and a small head cannot memorize pixel noise. A gradient-**detached** subtype head produces a weak incident-type hypothesis for Gemma's telemetry — it can never damage the binary objective.

High dropout doubles as **MC-dropout** at inference: the std of stochastic passes is an epistemic-uncertainty gate in §13.

In [ ]:
class SafetyFormer(nn.Module):
    """Window-level anomaly scorer: [N, L, D_in] -> (logit [N], subtype [N, C], attn [N, L])."""
    def __init__(self, d_in, cfg, n_subtypes):
        super().__init__()
        self.proj = nn.Sequential(nn.LayerNorm(d_in), nn.Linear(d_in, cfg.d_model))
        self.pos = nn.Parameter(torch.zeros(1, cfg.window_len, cfg.d_model))
        layer = nn.TransformerEncoderLayer(cfg.d_model, cfg.n_heads, 4 * cfg.d_model,
                                           cfg.dropout, batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, cfg.n_layers)
        self.query = nn.Parameter(torch.randn(1, 1, cfg.d_model) * 0.02)
        self.attn_pool = nn.MultiheadAttention(cfg.d_model, cfg.n_heads,
                                               dropout=cfg.dropout, batch_first=True)
        self.norm = nn.LayerNorm(cfg.d_model)
        self.score_head = nn.Linear(cfg.d_model, 1)
        self.subtype_head = nn.Linear(cfg.d_model, max(n_subtypes, 1))

    def forward(self, x):
        h = self.proj(x.float()) + self.pos[:, :x.size(1)]
        h = self.encoder(h)
        q = self.query.expand(h.size(0), -1, -1)
        pooled, attn_w = self.attn_pool(q, h, h, need_weights=True)
        z = self.norm(pooled.squeeze(1))
        # subtype head reads a DETACHED representation: auxiliary only, cannot
        # perturb the binary anomaly objective through shared weights
        return (self.score_head(z).squeeze(-1),
                self.subtype_head(z.detach()),
                attn_w.squeeze(1).squeeze(1))

def window_starts(n_frames, cfg):
    if n_frames <= cfg.window_len:
        return np.array([0])
    return np.arange(0, n_frames - cfg.window_len + 1, cfg.window_stride)

def get_window(feat, start, cfg):
    w = feat[start:start + cfg.window_len]
    if len(w) < cfg.window_len:                       # short video: edge-pad
        w = np.concatenate([w, np.repeat(w[-1:], cfg.window_len - len(w), axis=0)])
    return w

_probe = SafetyFormer(512, CFG, len(ANOMALY_CLASSES))
print(f"SafetyFormer trainable parameters: "
      f"{sum(p.numel() for p in _probe.parameters() if p.requires_grad):,}")
del _probe

## 10 · Top-k Multiple Instance Learning (the fix for weak labels)

**VIDEO = BAG, TEMPORAL WINDOWS = INSTANCES.** An "Assault" label says *at least some* windows are anomalous — not all of them. Forcing every window toward the video label (what the previous version of this project did) poisons training with mislabeled instances.

The MIL objective scores every window, then aggregates only the **top-k** most anomalous windows into the video-level score used by the binary loss:

```
video_logit = mean( top_k( window_logits ) ),   k = max(1, ceil(α · n_windows))
```

* **Anomalous videos:** gradient flows into the windows the model itself finds most suspicious — the anomalous segment — instead of every innocent window.
* **Normal videos:** the top-k are, by definition, the *highest*-scoring windows, so the loss actively pushes down exactly the windows most likely to cause false alarms.

α is tuned **on validation only** (§11). Loss is `BCEWithLogitsLoss` with `pos_weight` from the train distribution, A/B'd against binary focal loss on validation.

In [ ]:
class MILVideoDataset(Dataset):
    """Yields (windows [W, L, D] fp16, n_windows, binary label, subtype idx) per video."""
    def __init__(self, df, feats, cfg, train):
        self.rows = df.to_dict("records")
        self.feats, self.cfg, self.train = feats, cfg, train

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        r = self.rows[i]
        feat = self.feats[r["group"]]
        starts = window_starts(len(feat), self.cfg)
        if self.train and len(starts) > self.cfg.windows_per_video_train:
            starts = np.random.choice(starts, self.cfg.windows_per_video_train, replace=False)
        w = np.stack([get_window(feat, s, self.cfg) for s in starts])
        return (torch.from_numpy(w), len(starts),
                float(r["is_anomalous"]), int(r["subtype_idx"]))

def mil_collate(batch):
    Wmax = max(b[1] for b in batch)
    L, D = batch[0][0].shape[1], batch[0][0].shape[2]
    x = torch.zeros(len(batch), Wmax, L, D, dtype=torch.float16)
    mask = torch.zeros(len(batch), Wmax, dtype=torch.bool)
    for i, (w, n, _, _) in enumerate(batch):
        x[i, :n], mask[i, :n] = w, True
    y = torch.tensor([b[2] for b in batch])
    sub = torch.tensor([b[3] for b in batch])
    return x, mask, y, sub

def topk_video_logits(win_logits, mask, alpha):
    """win_logits [B, W] -> video logits [B] via masked top-k mean, and top-1 index.
    NOTE: mask is prefix-contiguous by collate construction, so indices into the
    masked vector coincide with indices into the padded row."""
    out, top1 = [], []
    for i in range(win_logits.size(0)):
        v = win_logits[i][mask[i]]
        k = max(1, math.ceil(alpha * v.numel()))
        topv, topi = torch.topk(v, k)
        out.append(topv.mean())
        top1.append(topi[0])
    return torch.stack(out), torch.stack(top1)

def binary_focal_loss(logits, targets, gamma=2.0, alpha=0.5):
    p = torch.sigmoid(logits)
    pt = torch.where(targets > 0.5, p, 1 - p)
    w = torch.where(targets > 0.5, torch.full_like(p, alpha), torch.full_like(p, 1 - alpha))
    return (-w * (1 - pt).pow(gamma) * torch.log(pt.clamp_min(1e-8))).mean()

@torch.no_grad()
def eval_video_scores(model, df, feats, cfg, alpha, chunk=512):
    """Video-level logits for a split (ALL windows, no sampling). Returns (logits, labels)."""
    model.eval()
    logits, labels = [], []
    for r in df.itertuples():
        feat = feats[r.group]
        starts = window_starts(len(feat), cfg)
        wl = []
        for s in range(0, len(starts), chunk):
            w = torch.from_numpy(np.stack(
                [get_window(feat, st, cfg) for st in starts[s:s + chunk]])).to(DEVICE)
            wl.append(model(w)[0].float().cpu())
        wl = torch.cat(wl)
        k = max(1, math.ceil(alpha * len(wl)))
        logits.append(wl.topk(k).values.mean().item())
        labels.append(float(r.is_anomalous))
    return np.array(logits), np.array(labels)

def train_safetyformer(tr_df, va_df, feats, d_in, cfg, alpha, loss_name, epochs, tag=""):
    """Train one configuration; early-stops and checkpoints on VALIDATION ROC-AUC."""
    seed_everything(cfg.seed)
    model = SafetyFormer(d_in, cfg, len(ANOMALY_CLASSES)).to(DEVICE)
    n_pos = max(int(tr_df.is_anomalous.sum()), 1)
    pos_weight = torch.tensor((len(tr_df) - n_pos) / n_pos).clamp(0.2, 5.0).to(DEVICE)
    loader = DataLoader(MILVideoDataset(tr_df, feats, cfg, train=True),
                        batch_size=cfg.batch_videos, shuffle=True,
                        collate_fn=mil_collate, num_workers=0, drop_last=False)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    total = max(epochs * len(loader), 1)
    warm = max(int(cfg.warmup_frac * total), 1)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda t: t / warm if t < warm
        else 0.02 + 0.98 * 0.5 * (1 + math.cos(math.pi * (t - warm) / max(total - warm, 1))))
    hist = {"loss": [], "val_auc": [], "val_ap": []}
    best = {"auc": -1, "state": None, "epoch": -1}
    step, bad = 0, 0
    for ep in range(epochs):
        model.train()
        ep_loss = []
        for x, mask, y, sub in loader:
            x, mask, y, sub = x.to(DEVICE), mask.to(DEVICE), y.to(DEVICE), sub.to(DEVICE)
            B, W, L, D = x.shape
            wl, st_logits, _ = model(x.view(B * W, L, D))
            wl = wl.view(B, W).masked_fill(~mask, -1e4)
            vid_logits, top1 = topk_video_logits(wl, mask, alpha)
            if loss_name == "focal":
                loss = binary_focal_loss(vid_logits, y)
            else:
                loss = F.binary_cross_entropy_with_logits(vid_logits, y, pos_weight=pos_weight)
            has_sub = sub >= 0
            if has_sub.any() and len(ANOMALY_CLASSES) > 1:   # weak subtype label on top-1 window
                st = st_logits.view(B, W, -1)[torch.arange(B, device=DEVICE), top1]
                loss = loss + cfg.subtype_loss_weight * F.cross_entropy(st[has_sub], sub[has_sub])
            opt.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            opt.step(); sched.step(); step += 1
            ep_loss.append(loss.item())
        v_logit, v_y = eval_video_scores(model, va_df, feats, cfg, alpha)
        auc = roc_auc_score(v_y, v_logit)
        ap = average_precision_score(v_y, v_logit)
        hist["loss"].append(float(np.mean(ep_loss)))
        hist["val_auc"].append(auc); hist["val_ap"].append(ap)
        marker = ""
        if auc > best["auc"]:
            best.update(auc=auc, ap=ap,
                        state={k: v.cpu().clone() for k, v in model.state_dict().items()},
                        epoch=ep)
            bad, marker = 0, "  <- best"
        else:
            bad += 1
        print(f"  [{tag}] epoch {ep+1:02d}/{epochs} loss={np.mean(ep_loss):.4f} "
              f"val ROC-AUC={auc:.4f} val PR-AUC={ap:.4f}{marker}")
        if bad >= cfg.patience:
            print(f"  [{tag}] early stop (no val ROC-AUC gain for {cfg.patience} epochs)")
            break
    model.load_state_dict(best["state"])
    return model, best, hist

print("MIL training machinery ready.")

## 11 · Backbone A/B, Experiment Table & Final Training

**Protocol (validation-only decisions, test untouched):**
1. **Backbone A/B** on a stratified 25% subset of train videos, both scored on the *full* validation split with an identical SafetyFormer config — winner by validation ROC-AUC.
2. **Experiment table** on the winner's full-train features: MIL α grid, then BCE-vs-focal at the best α. Plus a **baseline** (mean-pooled video embedding → logistic regression) to prove the temporal model earns its keep.
3. The best validation configuration is trained once on full data and **frozen**; the test set is evaluated exactly once, in §12.

In [ ]:
tr_df = manifest[manifest.split == "train"].reset_index(drop=True)
va_df = manifest[manifest.split == "val"].reset_index(drop=True)
te_df = manifest[manifest.split == "test"].reset_index(drop=True)

ab_tr, _ = train_test_split(tr_df, train_size=CFG.ab_train_frac,
                            stratify=tr_df.class_name, random_state=CFG.seed)
print(f"Backbone A/B: {len(ab_tr)} train videos (stratified {CFG.ab_train_frac:.0%} subset), "
      f"{len(va_df)} validation videos. Decision metric: validation ROC-AUC.\n")

ab_rows, ab_cache = [], {}
for tag, model_name in CFG.backbone_candidates:
    backbone, d_in, spec = build_backbone(model_name)
    feats = extract_features(pd.concat([ab_tr, va_df]).to_dict("records"), tag, backbone, spec)
    t0 = time.time()
    _, best, _ = train_safetyformer(ab_tr, va_df, feats, d_in, CFG,
                                    alpha=0.15, loss_name="bce",
                                    epochs=min(CFG.epochs, 12), tag=f"A/B {tag}")
    ab_rows.append(dict(backbone=tag, model=model_name, feat_dim=d_in,
                        val_roc_auc=round(best["auc"], 4), val_pr_auc=round(best["ap"], 4),
                        train_time_s=round(time.time() - t0, 1)))
    ab_cache[tag] = (model_name, d_in)
    del backbone
    torch.cuda.empty_cache()

ab_table = pd.DataFrame(ab_rows).sort_values("val_roc_auc", ascending=False)
print("\nBackbone A/B (identical head, identical validation split):")
print(ab_table.to_string(index=False))
BEST_BACKBONE = ab_table.iloc[0].backbone
RESULTS["backbone_ab"] = ab_table.to_dict("records")
print(f"\nSelected backbone: {BEST_BACKBONE} (validation ROC-AUC winner)")

In [ ]:
# ---- full feature extraction with the winning backbone (cache makes re-runs instant)
BB_NAME, FEAT_DIM = ab_cache[BEST_BACKBONE]
backbone, FEAT_DIM, BB_SPEC = build_backbone(BB_NAME)
FEATS = extract_features(manifest.to_dict("records"), BEST_BACKBONE, backbone, BB_SPEC)
manifest = manifest[manifest.group.isin(FEATS)].reset_index(drop=True)
tr_df = manifest[manifest.split == "train"].reset_index(drop=True)
va_df = manifest[manifest.split == "val"].reset_index(drop=True)
te_df = manifest[manifest.split == "test"].reset_index(drop=True)
n_win = sum(len(window_starts(len(FEATS[g]), CFG)) for g in manifest.group)
print(f"Cached embeddings for {len(FEATS)} videos "
      f"({sum(v.shape[0] for v in FEATS.values()):,} frames, dim {FEAT_DIM}) "
      f"-> {n_win:,} dense temporal windows.")
RESULTS["total_windows"] = int(n_win)

# ---- baseline: mean-pooled video embedding -> logistic regression
def _mean_embed(df):
    return np.stack([FEATS[g].astype(np.float32).mean(0) for g in df.group])
lr_clf = LogisticRegression(max_iter=2000, C=1.0)
lr_clf.fit(_mean_embed(tr_df), tr_df.is_anomalous.astype(int))
_p_val = lr_clf.predict_proba(_mean_embed(va_df))[:, 1]
exp_rows = [dict(experiment="baseline: mean-pool + logistic regression",
                 backbone=BEST_BACKBONE, mil_alpha=None, loss="logreg",
                 val_roc_auc=round(roc_auc_score(va_df.is_anomalous, _p_val), 4),
                 val_pr_auc=round(average_precision_score(va_df.is_anomalous, _p_val), 4),
                 train_time_s=None)]
print("Baseline:", exp_rows[0])

# ---- MIL alpha grid (BCE), then loss A/B at the best alpha — all decided on VALIDATION
run_store = {}
for alpha in CFG.mil_alpha_grid:
    t0 = time.time()
    model_a, best_a, hist_a = train_safetyformer(
        tr_df, va_df, FEATS, FEAT_DIM, CFG, alpha, "bce", CFG.epochs, tag=f"a={alpha}")
    exp_rows.append(dict(experiment=f"SafetyFormer top-k MIL (alpha={alpha}, BCE)",
                         backbone=BEST_BACKBONE, mil_alpha=alpha, loss="bce",
                         val_roc_auc=round(best_a["auc"], 4), val_pr_auc=round(best_a["ap"], 4),
                         train_time_s=round(time.time() - t0, 1)))
    run_store[("bce", alpha)] = (model_a, best_a, hist_a)

best_alpha = max(((r["mil_alpha"], r["val_roc_auc"]) for r in exp_rows if r["mil_alpha"]),
                 key=lambda t: t[1])[0]
t0 = time.time()
model_f, best_f, hist_f = train_safetyformer(
    tr_df, va_df, FEATS, FEAT_DIM, CFG, best_alpha, "focal", CFG.epochs, tag="focal")
exp_rows.append(dict(experiment=f"SafetyFormer top-k MIL (alpha={best_alpha}, focal)",
                     backbone=BEST_BACKBONE, mil_alpha=best_alpha, loss="focal",
                     val_roc_auc=round(best_f["auc"], 4), val_pr_auc=round(best_f["ap"], 4),
                     train_time_s=round(time.time() - t0, 1)))
run_store[("focal", best_alpha)] = (model_f, best_f, hist_f)

exp_table = pd.DataFrame(exp_rows)
print("\nEXPERIMENT TABLE (all selection on validation — test set untouched):")
print(exp_table.to_string(index=False))
RESULTS["experiments"] = exp_table.to_dict("records")

_mil = exp_table[exp_table.mil_alpha.notna()]
winner = _mil.sort_values("val_roc_auc", ascending=False).iloc[0]
MIL_ALPHA, LOSS_NAME = winner.mil_alpha, winner.loss
model, BEST, HIST = run_store[(LOSS_NAME, MIL_ALPHA)]
torch.save(model.state_dict(), Path(CFG.out_dir) / "safetyformer_best.pt")
print(f"\nFINAL CONFIG (frozen from validation): alpha={MIL_ALPHA}, loss={LOSS_NAME}, "
      f"val ROC-AUC={winner.val_roc_auc:.4f}")
RESULTS.update(mil_alpha=float(MIL_ALPHA), loss=LOSS_NAME,
               val_roc_auc=float(winner.val_roc_auc), val_pr_auc=float(winner.val_pr_auc))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(HIST["loss"], color="#3b528b", lw=2)
axes[0].set_title("Training loss (final configuration)")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("MIL loss")
axes[1].plot(HIST["val_auc"], color="#21918c", lw=2, label="val ROC-AUC")
axes[1].plot(HIST["val_ap"], color="#e45756", lw=2, ls="--", label="val PR-AUC")
axes[1].axvline(BEST["epoch"], color="gray", ls=":", label=f"best epoch ({BEST['epoch']+1})")
axes[1].set_title("Validation ROC-AUC / PR-AUC per epoch")
axes[1].set_xlabel("epoch"); axes[1].legend()
plt.tight_layout(); plt.show()

## 12 · Calibration, Operating Threshold & the One-Shot Test Evaluation

* **Temperature scaling** is fitted on **validation** video-level logits so that `P(anomalous)=0.8` means what it says — alarm policy on uncalibrated scores is policy on noise.
* **Operating threshold** — chosen on **validation** with a *recall-first triage rule*: among thresholds achieving recall ≥ 0.85, pick the one with the highest precision (fallback: max F1). Rationale: SafetyFormer is the *nomination* stage — missed incidents are unrecoverable, while false nominations are exactly what the downstream gates and Gemma 4 exist to absorb. The threshold is then **frozen** and applied unchanged to the test set.
* The **test set is evaluated exactly once**, below, with the frozen model, temperature, α, and threshold.

In [ ]:
class TemperatureScaler(nn.Module):
    def __init__(self):
        super().__init__()
        self.log_t = nn.Parameter(torch.zeros(1))
    def forward(self, logits):
        return logits / torch.exp(self.log_t)

val_logit, val_y = eval_video_scores(model, va_df, FEATS, CFG, MIL_ALPHA)
temp_scaler = TemperatureScaler()
_vl = torch.from_numpy(val_logit).float()
_vy = torch.from_numpy(val_y).float()
opt_t = torch.optim.LBFGS(temp_scaler.parameters(), lr=0.05, max_iter=100)
def _closure():
    opt_t.zero_grad()
    loss = F.binary_cross_entropy_with_logits(temp_scaler(_vl), _vy)
    loss.backward()
    return loss
opt_t.step(_closure)
TEMPERATURE = float(torch.exp(temp_scaler.log_t))
def calibrate(logits):
    return 1 / (1 + np.exp(-(np.asarray(logits) / TEMPERATURE)))

val_p_raw = 1 / (1 + np.exp(-val_logit))
val_p_cal = calibrate(val_logit)

def ece(p, y, bins=10):
    edges = np.linspace(0, 1, bins + 1)
    e = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (p > lo) & (p <= hi)
        if m.any():
            e += m.mean() * abs(p[m].mean() - y[m].mean())
    return e

print(f"Fitted temperature (validation only): T = {TEMPERATURE:.3f}")
print(f"Validation ECE  raw {ece(val_p_raw, val_y):.4f} -> calibrated {ece(val_p_cal, val_y):.4f}")
print(f"Validation Brier raw {brier_score_loss(val_y, val_p_raw):.4f} "
      f"-> calibrated {brier_score_loss(val_y, val_p_cal):.4f}")
RESULTS["temperature"] = round(TEMPERATURE, 3)
RESULTS["val_ece_calibrated"] = round(float(ece(val_p_cal, val_y)), 4)

fig, ax = plt.subplots(figsize=(5.5, 4.5))
for p, lab, c in [(val_p_raw, "raw", "#e45756"), (val_p_cal, "calibrated", "#21918c")]:
    edges = np.linspace(0, 1, 11)
    xs, ys = [], []
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (p > lo) & (p <= hi)
        if m.any():
            xs.append(p[m].mean()); ys.append(val_y[m].mean())
    ax.plot(xs, ys, "o-", color=c, label=lab)
ax.plot([0, 1], [0, 1], "k:", label="perfect")
ax.set_title("Reliability diagram (validation)")
ax.set_xlabel("predicted P(anomalous)"); ax.set_ylabel("observed anomaly rate")
ax.legend(); plt.tight_layout(); plt.show()

# ---- operating threshold: recall-first triage rule, VALIDATION ONLY, then frozen
prec_v, rec_v, thr_v = precision_recall_curve(val_y, val_p_cal)
ok = rec_v[:-1] >= CFG.val_min_recall
if ok.any():
    j = int(np.argmax(np.where(ok, prec_v[:-1], -1)))
    THRESH_RULE = f"max precision subject to validation recall >= {CFG.val_min_recall}"
else:
    f1s = 2 * prec_v[:-1] * rec_v[:-1] / np.clip(prec_v[:-1] + rec_v[:-1], 1e-9, None)
    j = int(np.argmax(f1s))
    THRESH_RULE = "max validation F1 (recall floor unattainable)"
ALARM_THRESHOLD = float(thr_v[j])
print(f"\nOperating threshold = {ALARM_THRESHOLD:.3f}  [rule: {THRESH_RULE}]")
print(f"  at this threshold (validation): precision={prec_v[j]:.3f} recall={rec_v[j]:.3f}")
RESULTS.update(alarm_threshold=round(ALARM_THRESHOLD, 3), threshold_rule=THRESH_RULE)

In [ ]:
# ================= ONE-SHOT TEST EVALUATION (frozen model / T / alpha / threshold) ==========
test_logit, test_y = eval_video_scores(model, te_df, FEATS, CFG, MIL_ALPHA)
test_p = calibrate(test_logit)
test_pred = (test_p >= ALARM_THRESHOLD).astype(int)

TEST_ROC_AUC = roc_auc_score(test_y, test_p)
TEST_PR_AUC = average_precision_score(test_y, test_p)
tn, fp, fn, tp = confusion_matrix(test_y, test_pred, labels=[0, 1]).ravel()
metrics = {
    "binary ROC-AUC (threshold-free)": round(TEST_ROC_AUC, 4),
    "binary PR-AUC / average precision": round(TEST_PR_AUC, 4),
    f"precision @ thr={ALARM_THRESHOLD:.3f}": round(precision_score(test_y, test_pred, zero_division=0), 4),
    f"recall    @ thr={ALARM_THRESHOLD:.3f}": round(recall_score(test_y, test_pred, zero_division=0), 4),
    f"F1        @ thr={ALARM_THRESHOLD:.3f}": round(f1_score(test_y, test_pred, zero_division=0), 4),
    "false-positive rate": round(fp / max(fp + tn, 1), 4),
}
print("TEST SET — evaluated exactly once with all decisions frozen from validation")
print("(metric names are exact: ROC-AUC is NOT accuracy)")
print("-" * 70)
for k, v in metrics.items():
    print(f"  {k:38s} {v}")
RESULTS["test_metrics"] = metrics

fig, axes = plt.subplots(1, 3, figsize=(16.5, 4.4))
fpr, tpr, _ = roc_curve(test_y, test_p)
axes[0].plot(fpr, tpr, color="#21918c", lw=2, label=f"ROC-AUC = {TEST_ROC_AUC:.3f}")
axes[0].plot([0, 1], [0, 1], "k:", lw=1)
axes[0].set_title("Test ROC — normal vs anomalous videos")
axes[0].set_xlabel("false-positive rate"); axes[0].set_ylabel("true-positive rate")
axes[0].legend()
pr_p, pr_r, _ = precision_recall_curve(test_y, test_p)
axes[1].plot(pr_r, pr_p, color="#3b528b", lw=2, label=f"PR-AUC = {TEST_PR_AUC:.3f}")
axes[1].axhline(test_y.mean(), color="gray", ls=":", label=f"prevalence {test_y.mean():.2f}")
axes[1].set_title("Test precision–recall")
axes[1].set_xlabel("recall"); axes[1].set_ylabel("precision"); axes[1].legend()
cm = confusion_matrix(test_y, test_pred, labels=[0, 1])
im = axes[2].imshow(cm, cmap="viridis")
for r in range(2):
    for c in range(2):
        axes[2].text(c, r, str(cm[r, c]), ha="center", va="center",
                     color="white" if cm[r, c] < cm.max() * 0.6 else "black", fontsize=13)
axes[2].set_xticks([0, 1], ["pred normal", "pred anomalous"])
axes[2].set_yticks([0, 1], ["true normal", "true anomalous"])
axes[2].set_title(f"Test confusion matrix @ thr={ALARM_THRESHOLD:.3f}")
plt.tight_layout(); plt.show()

# ---- diagnostic: do the two populations actually separate?
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(test_p[test_y == 0], bins=30, alpha=0.65, color="#21918c", label="normal videos")
ax.hist(test_p[test_y == 1], bins=30, alpha=0.65, color="#e45756", label="anomalous videos")
ax.axvline(ALARM_THRESHOLD, color="k", ls="--", label=f"threshold {ALARM_THRESHOLD:.2f}")
ax.set_title("Calibrated video anomaly score distributions (test)")
ax.set_xlabel("P(anomalous)"); ax.set_ylabel("videos"); ax.legend()
plt.tight_layout(); plt.show()

## 13 · False-Alarm Suppression Gates (pre-Gemma)

Raising an alarm on a single hot window *is* alarm fatigue. Before any candidate reaches Gemma, evidence must accumulate through independent gates:

1. **EMA smoothing** of calibrated window scores — kills single-window spikes;
2. **MC-dropout uncertainty rejection** — epistemically unstable windows can't vote;
3. **k-of-n persistence voting** — an incident must persist, not coincide;
4. **Hysteresis** — alarms raise high and clear low, eliminating boundary flicker.

These gates also implement the **selective-compute contract**: they decide *when Gemma 4 wakes up*, which we quantify as the Gemma activation rate in §17.

In [ ]:
def enable_mc_dropout(m):
    for mod in m.modules():
        if isinstance(mod, nn.Dropout):
            mod.train()

@torch.no_grad()
def window_scores_mc(model, feat, cfg, chunk=512):
    """Per-window calibrated probs + MC-dropout std for one video's features."""
    starts = window_starts(len(feat), cfg)
    outs = []
    for s in range(0, len(starts), chunk):
        w = torch.from_numpy(np.stack(
            [get_window(feat, st, cfg) for st in starts[s:s + chunk]])).to(DEVICE)
        runs = []
        model.eval(); enable_mc_dropout(model)
        for _ in range(cfg.mc_passes):
            runs.append(torch.sigmoid(model(w)[0].float() / TEMPERATURE).cpu().numpy())
        model.eval()
        runs = np.stack(runs)
        outs.append((runs.mean(0), runs.std(0)))
    p = np.concatenate([o[0] for o in outs])
    s = np.concatenate([o[1] for o in outs])
    return p, s, starts

class AlarmSuppressor:
    """EMA + uncertainty gate + k-of-n persistence + hysteresis. Stateful per stream."""
    def __init__(self, cfg, on_threshold):
        self.cfg, self.on = cfg, on_threshold
        self.off = on_threshold * cfg.sup_off_ratio
        self.reset()
    def reset(self):
        self.ema, self.alarm = 0.0, False
        self.votes = deque(maxlen=self.cfg.sup_window)
    def update(self, p, unc=0.0):
        c = self.cfg
        self.ema = c.sup_ema_alpha * p + (1 - c.sup_ema_alpha) * self.ema
        qualified = self.ema >= self.on and unc <= c.sup_std_max
        self.votes.append(int(qualified))
        persistence = sum(self.votes) / c.sup_window
        if not self.alarm and qualified and sum(self.votes) >= c.sup_k_required:
            self.alarm = True
        elif self.alarm and self.ema < self.off:
            self.alarm = False
        return {"p": p, "ema": self.ema, "qualified": qualified,
                "persistence": persistence, "alarm": self.alarm}

def run_gates(p_windows, std_windows, cfg, on_threshold):
    """Feed one video's window stream through the gates; return final state + trace."""
    sup = AlarmSuppressor(cfg, on_threshold)
    trace = [sup.update(p, u) for p, u in zip(p_windows, std_windows)]
    fired = any(t["alarm"] for t in trace)
    peak_pers = max(t["persistence"] for t in trace)
    return {"candidate": fired, "persistence": round(peak_pers, 3), "trace": trace}

# ---- compact stream simulation on held-out validation windows: naive vs gated
rng = np.random.default_rng(CFG.seed)
_norm_pool = [g for g, y in zip(va_df.group, va_df.is_anomalous) if not y]
_anom_pool = [g for g, y in zip(va_df.group, va_df.is_anomalous) if y]
_np_, _ns_, _ = window_scores_mc(model, FEATS[rng.choice(_norm_pool)], CFG)
_ap_, _as_, _ = window_scores_mc(model, FEATS[rng.choice(_anom_pool)], CFG)
stream_p = np.concatenate([_np_, _ap_, _np_[::-1]])
stream_s = np.concatenate([_ns_, _as_, _ns_[::-1]])
glitch = rng.choice(len(_np_), size=min(6, len(_np_)), replace=False)
stream_p[glitch] = rng.uniform(max(ALARM_THRESHOLD, 0.7), 0.98, glitch.size)  # sensor spikes
naive = stream_p >= ALARM_THRESHOLD
_sup = AlarmSuppressor(CFG, ALARM_THRESHOLD)
gated = np.array([_sup.update(p, u)["alarm"] for p, u in zip(stream_p, stream_s)])
event = np.zeros(len(stream_p), bool)
event[len(_np_):len(_np_) + len(_ap_)] = True
fig, ax = plt.subplots(figsize=(14, 3.6))
ax.plot(stream_p, color="#bbb", lw=1, label="calibrated window score")
ax.fill_between(range(len(event)), 0, event * 1.0, color="crimson", alpha=0.08,
                label="true anomalous video span")
ax.plot(naive * 0.98, color="#e45756", lw=1.4,
        label=f"naive threshold alarm ({int((naive & ~event).sum())} FP windows)")
ax.plot(gated * 1.02, color="#21918c", lw=2,
        label=f"gated alarm ({int((gated & ~event).sum())} FP windows)")
ax.set_title("Validation stream: injected single-window spikes fool a naive threshold; "
             "evidence-accumulation gates reject them")
ax.set_xlabel("analysis window"); ax.set_ylabel("P(anomalous)"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()
RESULTS["gate_sim"] = {"naive_fp_windows": int((naive & ~event).sum()),
                       "gated_fp_windows": int((gated & ~event).sum())}

## 14 · ★ Local Multimodal Gemma 4 Forensic Verifier ★

This is the load-bearing stage. **Gemma 4** (instruction-tuned, multimodal) runs **locally** — weights on this machine, no inference API, no frame ever leaves the device.

* **Checkpoint choice:** `google/gemma-4-E4B-it` — the E-series is Gemma 4's edge-first line (Per-Layer Embeddings, ~4.5B effective parameters, native image understanding with variable-resolution input). With 4-bit NF4 quantization it fits comfortably in a single 16 GB T4 — the same memory class as a Jetson-Orin edge box. `gemma-4-E2B-it` is the automatic smaller fallback.
* **Quantization:** bitsandbytes NF4 + double quantization; compute dtype fp16 on T4 (bf16 where supported).
* We print the GPU inventory before loading and the exact model ID, quantization config, device map, and memory footprint after — then measure real inference latency in §16/§17.

In [ ]:
# ---- GPU inventory BEFORE loading Gemma
print("GPU environment:")
if torch.cuda.is_available():
    print(f"  visible GPUs : {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        free_b, total_b = torch.cuda.mem_get_info(i)
        print(f"  GPU {i}: {p.name} | total {total_b/1e9:.1f} GB | free {free_b/1e9:.1f} GB")
else:
    print("  NO GPU — Gemma 4 local inference requires a GPU session on Kaggle.")

# Optional HF token (Gemma 4 is Apache-2.0; token only needed if the hub requires auth)
try:
    from kaggle_secrets import UserSecretsClient
    _tok = UserSecretsClient().get_secret("HF_TOKEN")
    if _tok:
        os.environ["HF_TOKEN"] = _tok
        print("HF_TOKEN loaded from Kaggle secrets.")
except Exception:
    pass

from transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig

gemma_model, gemma_processor, GEMMA_MODEL_ID, LOCAL_GEMMA = None, None, None, False
_compute_dtype = (torch.bfloat16 if torch.cuda.is_available()
                  and torch.cuda.is_bf16_supported() else torch.float16)
QUANT_CFG = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                               bnb_4bit_use_double_quant=True,
                               bnb_4bit_compute_dtype=_compute_dtype)
if torch.cuda.is_available():
    for _mid in CFG.gemma_candidates:
        try:
            print(f"\nLoading {_mid} (4-bit NF4, local weights) ...")
            gemma_processor = AutoProcessor.from_pretrained(_mid, padding_side="left")
            gemma_model = AutoModelForImageTextToText.from_pretrained(
                _mid, quantization_config=QUANT_CFG, device_map="auto",
                attn_implementation="sdpa")
            gemma_model.eval()
            GEMMA_MODEL_ID, LOCAL_GEMMA = _mid, True
            break
        except Exception as e:
            print(f"  could not load {_mid}: {type(e).__name__}: {str(e)[:200]}")

if LOCAL_GEMMA:
    _fp = getattr(gemma_model, "get_memory_footprint", lambda: 0)()
    print("\n" + "=" * 70)
    print("LOCAL GEMMA 4 ONLINE")
    print(f"  model ID          : {GEMMA_MODEL_ID}")
    print(f"  quantization      : 4-bit NF4, double-quant, compute dtype {_compute_dtype}")
    print(f"  device map        : {getattr(gemma_model, 'hf_device_map', 'auto')}")
    print(f"  memory footprint  : {_fp / 1e9:.2f} GB")
    print(f"  inference         : LOCAL (weights on this machine — no remote inference API)")
    print("=" * 70)
    RESULTS["gemma"] = {"model_id": GEMMA_MODEL_ID,
                        "quantization": f"4-bit NF4 (compute {str(_compute_dtype)})",
                        "memory_gb": round(_fp / 1e9, 2), "local_inference": True}
else:
    print("\n" + "!" * 70)
    print("!! GEMMA 4 UNAVAILABLE in this session (no GPU or download failure).      !!")
    print("!! The pipeline continues via the rule-based fallback, which is GRACEFUL  !!")
    print("!! DEGRADATION ONLY — every output it produces is labeled as fallback,    !!")
    print("!! never as Gemma. The submitted run uses local Gemma 4.                  !!")
    print("!" * 70)
    RESULTS["gemma"] = {"model_id": None, "local_inference": False}

## 15 · Evidence Selection, the Forensic Prompt & Robust JSON Parsing

**Evidence-aware keyframe selection** — Gemma never sees random frames. We locate the contiguous high-anomaly region around the peak window, sample candidate frames across its span, drop near-duplicates (mean absolute gray-level difference filter), and keep 4–8 temporally diverse frames. Each is anonymized *before* it becomes evidence.

**Prompt design** — short and adversarial by construction: Gemma is told the detector output is an **UNTRUSTED TRIAGE HYPOTHESIS** and that only visually supported findings count. One compact JSON schema; deterministic (greedy) decoding; a brace-aware parser that survives markdown fences, preamble text, and trailing commentary; one strict retry; then — only if parsing fails twice — an explicitly-labeled fallback.

In [ ]:
GEMMA_SYSTEM_PROMPT = (
    "You are the multimodal forensic verification stage of an edge public-safety system.\n"
    "A lightweight anomaly detector (SafetyFormer) has flagged a temporal window as a "
    "possible incident. You will see its telemetry, but treat it as an UNTRUSTED TRIAGE "
    "HYPOTHESIS: do not assume it is correct.\n"
    "Independently inspect the anonymized evidence frames (chronological order, faces "
    "pixelated for privacy). Use only visually supported evidence. Carefully distinguish "
    "genuine safety incidents from ordinary activity, crowds, fast but benign motion, "
    "camera artifacts, and ambiguous scenes. Low image quality alone is not evidence of "
    "an incident.\n"
    "Respond with ONE valid JSON object and nothing else, using exactly these keys:\n"
    '{"incident_supported": true|false, "incident_type": "<short label or none>", '
    '"severity": <int 1-5>, "visual_evidence": ["<observation>", ...], '
    '"false_positive_likelihood": <float 0-1>, "confidence": <float 0-1>, '
    '"recommended_action": "none"|"log_only"|"notify_operator"|"review_queue"|"urgent_dispatch"}'
)

ALLOWED_ACTIONS = ("none", "log_only", "notify_operator", "review_queue", "urgent_dispatch")

def extract_json(text):
    """Brace-aware JSON extraction: survives fences, preambles, trailing prose."""
    text = re.sub(r"```(?:json)?", " ", text)
    start = text.find("{")
    while start != -1:
        depth, in_str, esc = 0, False, False
        for i in range(start, len(text)):
            ch = text[i]
            if in_str:
                if esc:
                    esc = False
                elif ch == "\\":
                    esc = True
                elif ch == '"':
                    in_str = False
            elif ch == '"':
                in_str = True
            elif ch == "{":
                depth += 1
            elif ch == "}":
                depth -= 1
                if depth == 0:
                    try:
                        return json.loads(text[start:i + 1])
                    except json.JSONDecodeError:
                        break
        start = text.find("{", start + 1)
    return None

def validate_verdict(v):
    """Coerce types, clamp ranges; None if the object is unusable."""
    if not isinstance(v, dict):
        return None
    try:
        ev = v.get("visual_evidence", [])
        if isinstance(ev, str):
            ev = [ev]
        act = str(v.get("recommended_action", "review_queue")).lower().strip()
        return {
            "incident_supported": bool(v["incident_supported"]),
            "incident_type": str(v.get("incident_type", "unknown"))[:60],
            "severity": int(min(5, max(1, int(float(v.get("severity", 1)))))),
            "visual_evidence": [str(e)[:200] for e in ev][:8],
            "false_positive_likelihood": float(min(1.0, max(0.0, float(v["false_positive_likelihood"])))),
            "confidence": float(min(1.0, max(0.0, float(v["confidence"])))),
            "recommended_action": act if act in ALLOWED_ACTIONS else "review_queue",
        }
    except (KeyError, TypeError, ValueError):
        return None

def select_evidence(record, p_win, starts, cfg):
    """Keyframes from the contiguous high-anomaly region around the peak window,
    with near-duplicate suppression. Returns originals + anonymized + PIL for Gemma."""
    peak = int(np.argmax(p_win))
    lo = hi = peak
    floor = max(0.6 * p_win[peak], 0.05)
    while lo - 1 >= 0 and p_win[lo - 1] >= floor:
        lo -= 1
    while hi + 1 < len(p_win) and p_win[hi + 1] >= floor:
        hi += 1
    f0, f1 = int(starts[lo]), int(min(starts[hi] + cfg.window_len, len(record["feat_paths"])))
    cand = np.unique(np.linspace(f0, f1 - 1, cfg.n_evidence_frames * 3).round().astype(int))
    kept_idx, last_gray = [], None
    for i in cand:                                   # temporal-diversity / dedup filter
        fr = load_frame(record["feat_paths"][i])
        if fr is None:
            continue
        g = cv2.cvtColor(fr, cv2.COLOR_RGB2GRAY).astype(np.float32)
        if last_gray is None or np.abs(g - last_gray).mean() > 3.0:
            kept_idx.append(i)
            last_gray = g
    if len(kept_idx) < 2:                            # static scene: fall back to even spacing
        kept_idx = list(np.unique(np.linspace(f0, f1 - 1, cfg.n_evidence_frames).round().astype(int)))
    sel = np.unique(np.linspace(0, len(kept_idx) - 1,
                                min(cfg.n_evidence_frames, len(kept_idx))).round().astype(int))
    frame_idx = [kept_idx[i] for i in sel]
    orig = np.stack([load_frame(record["feat_paths"][i], size=cfg.evidence_size)
                     for i in frame_idx])
    anon, n_faces = anonymizer.anonymize_clip(orig)
    return {"frame_idx": frame_idx, "orig": orig, "anon": anon,
            "pil": [Image.fromarray(f) for f in anon],
            "region": (lo, hi), "span_frames": (f0, f1), "faces_pixelated": n_faces}

def build_telemetry(p_cal, subtype, uncertainty, persistence, n_frames, cfg):
    return (
        "UNTRUSTED TRIAGE HYPOTHESIS from the lightweight detector — verify independently:\n"
        f"- calibrated anomaly score: {p_cal:.2f} (operating threshold {ALARM_THRESHOLD:.2f})\n"
        f"- top incident-type hypothesis: {subtype}\n"
        f"- detector epistemic uncertainty (MC-dropout std): {uncertainty:.3f}\n"
        f"- temporal persistence (fraction of recent windows voting anomalous): {persistence:.2f}\n"
        f"- the {n_frames} frames span the highest-anomaly temporal region, chronological, "
        "faces pixelated on-device.\n"
        "Inspect the frames and return ONLY the JSON object."
    )

GEMMA_LATENCIES = []

class RuleBasedVerifier:
    """GRACEFUL DEGRADATION ONLY (no visual inspection). Output is explicitly labeled
    as fallback — it is never presented as Gemma."""
    def verify(self, pil_frames, telemetry, tag=""):
        m = re.search(r"anomaly score: ([\d.]+)", telemetry)
        p = float(m.group(1)) if m else 0.5
        v = {"incident_supported": p >= 0.75, "incident_type": "unverified",
             "severity": 3 if p >= 0.75 else 1,
             "visual_evidence": ["FALLBACK: no visual inspection performed"],
             "false_positive_likelihood": round(1 - p, 2), "confidence": 0.30,
             "recommended_action": "review_queue"}
        return v, {"verifier": "rule_based_fallback (graceful degradation — NOT Gemma)",
                   "latency_s": 0.0, "raw": ""}

class GemmaForensicVerifier:
    """Local multimodal Gemma 4: anonymized frames in -> validated JSON verdict out."""
    def __init__(self, model, processor, model_id, cfg):
        self.model, self.processor, self.model_id, self.cfg = model, processor, model_id, cfg
        self.fallback = RuleBasedVerifier()

    def _generate(self, pil_frames, text):
        messages = [
            {"role": "system", "content": [{"type": "text", "text": GEMMA_SYSTEM_PROMPT}]},
            {"role": "user", "content":
                [{"type": "image", "image": im} for im in pil_frames]
                + [{"type": "text", "text": text}]},
        ]
        try:
            inputs = self.processor.apply_chat_template(
                messages, tokenize=True, return_dict=True, return_tensors="pt",
                add_generation_prompt=True)
        except Exception:                       # processor variant that wants image paths
            tmp = Path(CFG.out_dir) / "gemma_tmp"
            tmp.mkdir(exist_ok=True)
            for m in messages[1]["content"]:
                if m.get("type") == "image":
                    p = tmp / f"ev_{id(m)}.jpg"
                    m["image"].save(p)
                    m.pop("image"); m["path"] = str(p)
            inputs = self.processor.apply_chat_template(
                messages, tokenize=True, return_dict=True, return_tensors="pt",
                add_generation_prompt=True)
        inputs = inputs.to(self.model.device)
        n_in = inputs["input_ids"].shape[-1]
        t0 = time.perf_counter()
        with torch.inference_mode():
            out = self.model.generate(**inputs, max_new_tokens=self.cfg.gemma_max_new_tokens,
                                      do_sample=False)
        dt = time.perf_counter() - t0
        new_tokens = out[0][n_in:]
        text_out = self.processor.decode(new_tokens, skip_special_tokens=True)
        torch.cuda.empty_cache()
        return text_out, dt, int(len(new_tokens))

    def verify(self, pil_frames, telemetry, tag=""):
        attempt_text = telemetry
        for attempt in range(2):
            raw, dt, n_tok = self._generate(pil_frames, attempt_text)
            GEMMA_LATENCIES.append({"tag": tag, "latency_s": dt, "new_tokens": n_tok,
                                    "tok_per_s": n_tok / max(dt, 1e-6)})
            verdict = validate_verdict(extract_json(raw))
            if verdict is not None:
                return verdict, {"verifier": f"{self.model_id} (LOCAL, 4-bit NF4)",
                                 "latency_s": round(dt, 2), "raw": raw}
            attempt_text = telemetry + "\nYour previous reply was not valid JSON. Return ONLY the JSON object."
        print("  Gemma returned unparseable JSON twice -> explicit fallback engaged.")
        return self.fallback.verify(pil_frames, telemetry, tag)

verifier = (GemmaForensicVerifier(gemma_model, gemma_processor, GEMMA_MODEL_ID, CFG)
            if LOCAL_GEMMA else RuleBasedVerifier())
print(f"Active verifier: "
      f"{GEMMA_MODEL_ID + ' (local multimodal Gemma 4)' if LOCAL_GEMMA else 'rule-based fallback (degraded)'}")

## 16 · SafetyFormer × Gemma Decision Fusion

The final decision is made by an explicit, five-rule engine — no hidden logic. Gemma can veto or downgrade the detector, but it cannot *silently* discard overwhelming temporal evidence (rule R2 routes that conflict to a human):

| Rule | Condition | Decision |
|---|---|---|
| **R1** | detector ≥ threshold **and** Gemma confirms (confidence ≥ 0.6, FP-likelihood ≤ 0.4) | **ESCALATE** |
| **R2** | Gemma rejects **but** detector is extreme (P ≥ 0.92, persistence ≥ 0.6) | **REVIEW** (never silent discard) |
| **R3** | Gemma rejects with FP-likelihood ≥ 0.6 | **SUPPRESS** |
| **R4** | Gemma supports but with reservations | **REVIEW** |
| **R5** | anything else (ambiguous) | **REVIEW** |

In [ ]:
class IncidentDecisionEngine:
    """Transparent fusion of detector evidence and Gemma's visual verification."""
    def __init__(self, threshold):
        self.thr = threshold

    def decide(self, p_cal, persistence, uncertainty, verdict):
        g = verdict
        if (g["incident_supported"] and g["confidence"] >= 0.60
                and g["false_positive_likelihood"] <= 0.40 and p_cal >= self.thr):
            return "ESCALATE", ("R1: detector above threshold AND Gemma visually confirms "
                                f"(confidence {g['confidence']:.2f}, "
                                f"FP-likelihood {g['false_positive_likelihood']:.2f})")
        if not g["incident_supported"] and p_cal >= 0.92 and persistence >= 0.60:
            return "REVIEW", ("R2: Gemma rejects, but temporal evidence is extreme "
                              f"(P={p_cal:.2f}, persistence={persistence:.2f}) — "
                              "routed to a human, never silently discarded")
        if not g["incident_supported"] and g["false_positive_likelihood"] >= 0.60:
            return "SUPPRESS", ("R3: Gemma finds no visual support and rates this a likely "
                                f"false positive ({g['false_positive_likelihood']:.2f})")
        if g["incident_supported"]:
            return "REVIEW", ("R4: Gemma sees a possible incident but with reservations "
                              f"(confidence {g['confidence']:.2f})")
        return "REVIEW", "R5: ambiguous evidence — conservative human review"

decision_engine = IncidentDecisionEngine(ALARM_THRESHOLD)
print(f"IncidentDecisionEngine ready (detector threshold {ALARM_THRESHOLD:.3f}).")

## 17 · END-TO-END GEMMA 4 INCIDENT VERIFICATION DEMO

The complete edge pipeline on real held-out test videos, **from raw frames** (nothing precomputed in this section is reused for scoring — embeddings are recomputed live to prove the pipeline works end-to-end):

frames → anonymize → embed → SafetyFormer windows → gates → evidence keyframes → **local Gemma 4** → fusion → incident card → encrypted dispatch.

Each case shows the anomaly timeline, the exact anonymized frames Gemma inspected, Gemma's verbatim JSON verdict, and the final decision.

In [ ]:
def print_incident_card(rep):
    W = 74
    def row(k, v):
        print(f"| {k:<26s} {str(v)[:W-31]:<{W-31}s} |")
    bar = "+" + "-" * (W - 2) + "+"
    print(bar)
    print(f"| {'AEGIS INCIDENT REPORT — ' + rep['sequence'][:W-28]:<{W-4}s} |")
    print(bar)
    row("ground truth (hidden)", rep["ground_truth"])
    print(f"| {'SAFETYFORMER TRIAGE':<{W-4}s} |")
    row("  anomaly score (calibrated)", f"{rep['p_cal']:.3f}  (threshold {ALARM_THRESHOLD:.3f})")
    row("  temporal persistence", f"{rep['persistence']:.2f}")
    row("  uncertainty (MC std)", f"{rep['uncertainty']:.3f}")
    row("  subtype hypothesis", rep["subtype_hypothesis"])
    print(f"| {'GEMMA 4 VISUAL VERIFICATION':<{W-4}s} |")
    row("  verifier", rep["verifier"])
    row("  incident supported", rep["verdict"]["incident_supported"])
    row("  incident type", rep["verdict"]["incident_type"])
    row("  severity (1-5)", rep["verdict"]["severity"])
    row("  confidence", f"{rep['verdict']['confidence']:.2f}")
    row("  FP likelihood", f"{rep['verdict']['false_positive_likelihood']:.2f}")
    for i, e in enumerate(rep["verdict"]["visual_evidence"][:4]):
        row(f"  evidence {i+1}", e)
    print(f"| {'FINAL EDGE DECISION':<{W-4}s} |")
    row("  decision", rep["decision"])
    row("  rationale", rep["rationale"])
    row("  recommended response", rep["verdict"]["recommended_action"])
    print(f"| {'PRIVACY STATUS':<{W-4}s} |")
    row("  faces anonymized pre-Gemma", f"yes ({rep['faces_pixelated']} region(s) pixelated)")
    row("  Gemma inference", "LOCAL" if LOCAL_GEMMA else "FALLBACK (degraded)")
    row("  raw video uploaded to cloud", "NO")
    row("  encrypted dispatch payload", f"{rep['dispatch_kb']:.1f} KB (AES-256-GCM)")
    print(bar)

def run_incident_pipeline(record, force_candidate=False, show=True, use_cache=False):
    """Full edge pipeline for one video. force_candidate lets us demo Gemma's veto on
    near-miss normal videos even when the gates would already have stopped them."""
    t, rep = {}, {"sequence": record["group"], "ground_truth": record["class_name"]}
    t0 = time.perf_counter()
    if use_cache and record["group"] in FEATS:
        feat = FEATS[record["group"]]
    else:                                          # live path: prove it works from pixels
        frames = [f for f in (load_frame(p) for p in record["feat_paths"]) if f is not None]
        t["frame_io_s"] = round(time.perf_counter() - t0, 2)
        t0 = time.perf_counter()
        feat = embed_frames_now(frames, backbone, BB_SPEC)
        t["embedding_s"] = round(time.perf_counter() - t0, 2)
    t0 = time.perf_counter()
    p_win, std_win, starts = window_scores_mc(model, feat, CFG)
    k = max(1, math.ceil(MIL_ALPHA * len(p_win)))
    p_cal = float(np.sort(p_win)[-k:].mean())      # top-k mean of calibrated window probs
    gates = run_gates(p_win, std_win, CFG, ALARM_THRESHOLD)
    t["triage_gates_s"] = round(time.perf_counter() - t0, 2)
    top_w = int(np.argmax(p_win))
    with torch.no_grad():
        _, st_logits, _ = model(torch.from_numpy(
            get_window(feat, int(starts[top_w]), CFG)).unsqueeze(0).to(DEVICE))
    subtype = (ANOMALY_CLASSES[int(st_logits[0].argmax())]
               if ANOMALY_CLASSES else "unknown")
    rep.update(p_cal=round(p_cal, 3), persistence=gates["persistence"],
               uncertainty=round(float(std_win.max()), 3), subtype_hypothesis=subtype,
               candidate=gates["candidate"])
    if not (gates["candidate"] or force_candidate):
        rep.update(decision="NO CANDIDATE", verdict=None,
                   rationale="gates never fired — Gemma was not activated (selective compute)")
        if show:
            print(f"[{record['group']}] no candidate incident; Gemma not woken. "
                  f"(P={p_cal:.2f}, persistence={gates['persistence']:.2f})")
        return rep
    # ---- evidence -> Gemma
    t0 = time.perf_counter()
    ev = select_evidence(record, p_win, starts, CFG)
    telemetry = build_telemetry(p_cal, subtype, float(std_win.max()),
                                gates["persistence"], len(ev["pil"]), CFG)
    verdict, meta = verifier.verify(ev["pil"], telemetry, tag=record["group"])
    t["gemma_verification_s"] = round(time.perf_counter() - t0, 2)
    decision, rationale = decision_engine.decide(p_cal, gates["persistence"],
                                                 float(std_win.max()), verdict)
    # ---- encrypted dispatch payload (final report + anonymized evidence JPEGs)
    jpgs = [np.frombuffer(cv2.imencode(".jpg", cv2.cvtColor(f, cv2.COLOR_RGB2BGR))[1]
                          .tobytes(), dtype=np.uint8) for f in ev["anon"]]
    blob = channel.encrypt({"report": json.dumps({**verdict, "decision": decision,
                                                  "sequence": record["group"]}),
                            **{f"evidence_{i}": j for i, j in enumerate(jpgs)}})
    rep.update(verdict=verdict, verifier=meta["verifier"], decision=decision,
               rationale=rationale, faces_pixelated=ev["faces_pixelated"],
               dispatch_kb=len(blob) / 1024, stage_latency=t,
               gemma_raw=meta.get("raw", ""))
    if show:
        fig, ax = plt.subplots(figsize=(13, 3.2))
        ax.plot(p_win, color="#3b528b", lw=1.6, label="calibrated window score")
        ax.axhline(ALARM_THRESHOLD, color="#e45756", ls="--", lw=1,
                   label=f"threshold {ALARM_THRESHOLD:.2f}")
        lo, hi = ev["region"]
        ax.axvspan(lo, hi, color="crimson", alpha=0.15, label="selected evidence region")
        ev_w = [np.searchsorted(starts, i, side="right") - 1 for i in ev["frame_idx"]]
        ax.scatter(ev_w, [p_win[min(max(w, 0), len(p_win)-1)] for w in ev_w],
                   color="crimson", zorder=5, s=28, label="evidence keyframes")
        ax.set_title(f"{record['group']} — window-level anomaly timeline "
                     f"(truth: {record['class_name']})")
        ax.set_xlabel("temporal window (~5 s each)"); ax.set_ylabel("P(anomalous)")
        ax.legend(fontsize=8, ncol=4)
        plt.tight_layout(); plt.show()
        n = len(ev["pil"])
        fig, axes = plt.subplots(2, n, figsize=(2.1 * n, 4.6))
        axes = np.asarray(axes).reshape(2, -1)     # robust when n == 1
        for i in range(n):
            axes[0, i].imshow(ev["orig"][i]); axes[1, i].imshow(ev["anon"][i])
            axes[0, i].set_axis_off(); axes[1, i].set_axis_off()
            axes[1, i].set_title(f"frame {ev['frame_idx'][i]}", fontsize=7, y=-0.18)
        axes[0, 0].set_title("ORIGINAL (device-only)", fontsize=8, loc="left")
        axes[1, 0].set_title("ANONYMIZED — exactly what Gemma 4 inspected", fontsize=8, loc="left")
        plt.suptitle("Evidence panel: flagged region -> anonymized -> Gemma input", y=1.0)
        plt.tight_layout(); plt.show()
        print(f"Gemma verdict [{meta['verifier']}] "
              f"(latency {meta.get('latency_s', 0)}s):")
        print(json.dumps(verdict, indent=2))
        print()
        print_incident_card(rep)
    return rep

print("End-to-end pipeline assembled.")

In [ ]:
# ---- pick real case studies from the held-out TEST set (no cherry-picked fakes):
#  (a) anomalous videos the detector flags  -> Gemma should confirm  -> ESCALATE
#  (b) the highest-scoring NORMAL videos    -> natural false positives -> Gemma should veto
_p_by_group = dict(zip(te_df.group, test_p))
_anom_sorted = te_df[te_df.is_anomalous].assign(p=lambda d: d.group.map(_p_by_group))\
                    .sort_values("p", ascending=False)
_norm_sorted = te_df[~te_df.is_anomalous].assign(p=lambda d: d.group.map(_p_by_group))\
                    .sort_values("p", ascending=False)

DEMO_REPORTS = []
print("=" * 78)
print("CASE STUDIES A — anomalous test videos (expected: Gemma confirms -> ESCALATE)")
print("=" * 78)
for _, rec in _anom_sorted.head(CFG.demo_cases).iterrows():
    DEMO_REPORTS.append(run_incident_pipeline(rec.to_dict(), force_candidate=True, show=True))

print("\n" + "=" * 78)
print("CASE STUDIES B — highest-scoring NORMAL test videos (the detector's natural")
print("false positives; expected: Gemma finds no visual support -> SUPPRESS/REVIEW)")
print("=" * 78)
for _, rec in _norm_sorted.head(CFG.demo_cases).iterrows():
    DEMO_REPORTS.append(run_incident_pipeline(rec.to_dict(), force_candidate=True, show=True))

RESULTS["demo_cases"] = [{k: r.get(k) for k in
                          ("sequence", "ground_truth", "p_cal", "decision", "rationale")}
                         for r in DEMO_REPORTS]

## 18 · Ablation — What Does Gemma 4 Actually Add?

Stages are cumulative; each row must earn its place:

* **A. SafetyFormer only** — calibrated score vs frozen threshold (full test set, video-level).
* **B. + suppression gates** — persistence/uncertainty/hysteresis must also agree.
* **C. + Gemma 4 visual verification** — Gemma inspects evidence frames; the fusion engine makes the final call. An alarm here means **ESCALATE**; REVIEW is reported separately (it is a human-queue outcome, not a dispatch).

**Honest protocol note:** stages A/B are evaluated on the full test set. Stage C requires one Gemma inference per candidate, so it runs on a **clearly-labeled stratified random subset** of test videos (size printed below, seeded) — we report exactly what was evaluated. Gemma latency statistics are measured on these real calls.

In [ ]:
rng = np.random.default_rng(CFG.seed)
n_a = min(CFG.ablation_videos_per_class, int(te_df.is_anomalous.sum()))
n_n = min(CFG.ablation_videos_per_class, int((~te_df.is_anomalous).sum()))
sub_df = pd.concat([
    te_df[te_df.is_anomalous].sample(n_a, random_state=CFG.seed),
    te_df[~te_df.is_anomalous].sample(n_n, random_state=CFG.seed),
]).reset_index(drop=True)
print(f"Gemma-evaluation subset: {n_a} anomalous + {n_n} normal TEST videos "
      f"(seeded stratified sample — NOT the full test set; stages A/B use the full test set).")

# ---- stages A and B on the FULL test set
stageA_alarm = (test_p >= ALARM_THRESHOLD)
gate_pass_full = {}
for r in tqdm(te_df.itertuples(), total=len(te_df), desc="gates (full test)"):
    p_w, s_w, _ = window_scores_mc(model, FEATS[r.group], CFG)
    gate_pass_full[r.group] = run_gates(p_w, s_w, CFG, ALARM_THRESHOLD)["candidate"]
stageB_alarm = np.array([_p >= ALARM_THRESHOLD and gate_pass_full[g]
                         for g, _p in zip(te_df.group, test_p)])

def stage_metrics(alarms, ys):
    alarms, ys = np.asarray(alarms, bool), np.asarray(ys, bool)
    tp, fp = int((alarms & ys).sum()), int((alarms & ~ys).sum())
    fn = int((~alarms & ys).sum())
    return dict(TP=tp, FP=fp, FN=fn,
                precision=round(tp / max(tp + fp, 1), 3),
                recall=round(tp / max(tp + fn, 1), 3))

rows = [dict(stage="A. SafetyFormer only (full test)", n=len(te_df),
             **stage_metrics(stageA_alarm, test_y.astype(bool)), REVIEW=0),
        dict(stage="B. + suppression gates (full test)", n=len(te_df),
             **stage_metrics(stageB_alarm, test_y.astype(bool)), REVIEW=0)]

# ---- stage C on the labeled subset: Gemma verifies every stage-B candidate
sub_y, sub_A, sub_C, n_review, gemma_calls = [], [], [], 0, 0
subset_windows = 0
for r in sub_df.itertuples():
    p_v = _p_by_group[r.group]
    p_w, s_w, starts = window_scores_mc(model, FEATS[r.group], CFG)
    subset_windows += len(p_w)
    a_alarm = p_v >= ALARM_THRESHOLD
    b_alarm = a_alarm and gate_pass_full.get(r.group, run_gates(p_w, s_w, CFG,
                                                                ALARM_THRESHOLD)["candidate"])
    c_alarm = False
    if b_alarm:                                  # only candidates wake Gemma
        gemma_calls += 1
        rep = run_incident_pipeline(r._asdict() | {"group": r.group}, show=False,
                                    use_cache=True, force_candidate=True)
        c_alarm = rep["decision"] == "ESCALATE"
        n_review += int(rep["decision"] == "REVIEW")
    sub_y.append(bool(r.is_anomalous)); sub_A.append(a_alarm); sub_C.append(c_alarm)

sub_B = [a and gate_pass_full[g] for a, g in zip(sub_A, sub_df.group)]
rows += [dict(stage="A. SafetyFormer only (subset)", n=len(sub_df),
              **stage_metrics(sub_A, sub_y), REVIEW=0),
         dict(stage="B. + gates (subset)", n=len(sub_df),
              **stage_metrics(sub_B, sub_y), REVIEW=0),
         dict(stage=("C. + Gemma 4 visual verification (subset)" if LOCAL_GEMMA
                     else "C. + FALLBACK verifier — NOT Gemma (subset)"), n=len(sub_df),
              **stage_metrics(sub_C, sub_y), REVIEW=n_review)]

abl_table = pd.DataFrame(rows)
print("\nABLATION (alarm = dispatch-grade alarm; REVIEW = routed to human queue):")
print(abl_table.to_string(index=False))
RESULTS["ablation"] = abl_table.to_dict("records")

_subA, _subC = stage_metrics(sub_A, sub_y), stage_metrics(sub_C, sub_y)
if _subA["FP"] > 0:
    print(f"\nFalse-positive dispatches on the subset: {_subA['FP']} -> {_subC['FP']} "
          f"({(1 - _subC['FP'] / _subA['FP']) * 100:.0f}% reduction) while recall went "
          f"{_subA['recall']:.2f} -> {_subC['recall']:.2f} "
          f"(+{n_review} case(s) routed to human review).")
    RESULTS["fp_reduction_pct"] = round((1 - _subC["FP"] / _subA["FP"]) * 100, 1)
RESULTS["subset_recall_A"], RESULTS["subset_recall_C"] = _subA["recall"], _subC["recall"]

fig, ax = plt.subplots(figsize=(8.5, 3.8))
sub_rows = abl_table[abl_table.stage.str.contains("subset")]
x = np.arange(len(sub_rows))
ax.bar(x - 0.18, sub_rows.FP, 0.36, color="#e45756", label="false-positive dispatches")
ax.bar(x + 0.18, sub_rows.TP, 0.36, color="#21918c", label="true-positive dispatches")
ax.set_xticks(x, ["A\ndetector", "B\n+gates", "C\n+Gemma 4"], fontsize=9)
ax.set_title(f"Ablation on the {len(sub_df)}-video labeled test subset")
ax.set_ylabel("videos"); ax.legend()
plt.tight_layout(); plt.show()

# ---- Gemma latency (measured on the real calls above + the demo)
if GEMMA_LATENCIES:
    lat = np.array([g["latency_s"] for g in GEMMA_LATENCIES])
    tps = np.array([g["tok_per_s"] for g in GEMMA_LATENCIES])
    RESULTS["gemma_latency"] = {
        "n_calls": len(lat), "mean_s": round(float(lat.mean()), 2),
        "median_s": round(float(np.median(lat)), 2),
        "p95_s": round(float(np.percentile(lat, 95)), 2) if len(lat) >= 10 else None,
        "throughput_tok_per_s": round(float(tps.mean()), 1)}
    print(f"\nGemma 4 inference latency over {len(lat)} real local calls: "
          f"mean {lat.mean():.2f}s | median {np.median(lat):.2f}s"
          + (f" | p95 {np.percentile(lat, 95):.2f}s" if len(lat) >= 10 else "")
          + f" | {tps.mean():.1f} tok/s")
else:
    print("\nGemma latency: not measured (no local Gemma calls this session).")

# ---- selective-compute accounting
if gemma_calls:
    act_rate = gemma_calls / max(subset_windows, 1)
    RESULTS["selective_compute"] = {
        "subset_windows_analyzed": int(subset_windows),
        "gemma_calls": int(gemma_calls),
        "activation_rate": round(act_rate, 5),
        "vlm_calls_avoided_vs_per_window": int(subset_windows - gemma_calls),
        "vlm_reduction_pct": round((1 - gemma_calls / max(subset_windows, 1)) * 100, 2)}
    print(f"\nSELECTIVE COMPUTE: {subset_windows:,} temporal windows analyzed on the subset; "
          f"only {gemma_calls} candidate incidents woke Gemma "
          f"-> activation rate {act_rate:.4f} "
          f"({(1 - act_rate) * 100:.1f}% of per-window VLM calls avoided).")

In [ ]:
# ==== Bridge API: exposes the REAL pipeline (already loaded in memory) to Streamlit ====
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "flask"])

from flask import Flask, request, jsonify
import threading, tempfile, traceback, time

bridge = Flask(__name__)

@bridge.route("/analyze", methods=["POST"])
def analyze():
    try:
        f = request.files["video"]
        tmp_path = tempfile.NamedTemporaryFile(delete=False, suffix=".mp4").name
        f.save(tmp_path)

        record = {
            "group": f"live_upload_{int(time.time())}",
            "class_name": "unknown",
            "kind": "video",
            "feat_paths": probe_video(tmp_path, CFG),
        }
        if len(record["feat_paths"]) < 4:
            return jsonify({"error": "video too short/unreadable"}), 400

        # force_candidate=True so the real Gemma verifier always runs and inspects
        # the actual uploaded frames, even if the gates alone wouldn't have fired
        rep = run_incident_pipeline(record, force_candidate=True, show=False, use_cache=False)

        return jsonify({
            "p_cal": rep.get("p_cal"), "persistence": rep.get("persistence"),
            "uncertainty": rep.get("uncertainty"),
            "subtype_hypothesis": rep.get("subtype_hypothesis"),
            "decision": rep.get("decision"), "rationale": rep.get("rationale"),
            "verifier": rep.get("verifier"), "verdict": rep.get("verdict"),
            "faces_pixelated": rep.get("faces_pixelated"),
            "dispatch_kb": rep.get("dispatch_kb"),
        })
    except Exception as e:
        traceback.print_exc()
        return jsonify({"error": str(e)}), 500

threading.Thread(target=lambda: bridge.run(host="127.0.0.1", port=5005,
                                            debug=False, use_reloader=False),
                  daemon=True).start()
time.sleep(1)
print("Bridge live at http://127.0.0.1:5005/analyze — calls the REAL SafetyFormer + local Gemma 4 pipeline.")

## 19 · Privacy & Edge Feasibility

**Privacy scope, stated precisely:** faces detectable at source resolution are irreversibly pixelated *before* Gemma inspection; bodies/clothing/scene context intentionally remain (responders need them); Gemma weights execute locally so neither raw frames nor prompts transit any network; only the AES-256-GCM-encrypted incident report + selected anonymized evidence leave the device. This is strong *practical* privacy — **not** a formal de-identification guarantee, and we do not claim one.

In [ ]:
sf_params = sum(p.numel() for p in model.parameters())
bb_params = sum(p.numel() for p in backbone.parameters())
print("EDGE FEASIBILITY (measured on this session's hardware)")
print("-" * 66)
print(f"  frozen backbone ({BB_SPEC['name']}): {bb_params/1e6:.1f}M params (fp16 ~{bb_params*2/1e9:.2f} GB)")
print(f"  SafetyFormer triage head: {sf_params/1e6:.2f}M params — negligible")
if LOCAL_GEMMA:
    print(f"  Gemma 4 verifier: {GEMMA_MODEL_ID}, 4-bit NF4, "
          f"{RESULTS['gemma']['memory_gb']:.2f} GB resident")
if RESULTS.get("gemma_latency"):
    gl = RESULTS["gemma_latency"]
    print(f"  Gemma per-incident latency: mean {gl['mean_s']}s (median {gl['median_s']}s) — "
          "acceptable because it runs per CANDIDATE, not per frame")
_last = next((r for r in DEMO_REPORTS if r.get("stage_latency")), None)
if _last:
    print(f"  demo per-stage latency (s): {_last['stage_latency']}")
print(f"  dispatch payload: {_last['dispatch_kb']:.1f} KB encrypted "
      f"(vs streaming raw video continuously)" if _last else "")
print("-" * 66)
print("A 16 GB T4 hosts the entire stack simultaneously — the same memory class as a")
print("Jetson-Orin-family edge box. Continuous cost is SafetyFormer-only; Gemma 4 wakes")
print("selectively (activation rate above).")

## 20 · Limitations

1. **UCF-Crime is a prototype benchmark** — weakly-labeled, low-resolution, single-camera. Real deployment demands domain-specific data, threat modeling, and validation; nothing here is deployment-ready.
2. **Anonymization is best-effort**, not formal de-identification: Haar-based face detection misses profile/occluded/low-res faces, and gait/clothing remain identifying. A production system would use a stronger on-device detector (and possibly full-body redaction policies).
3. **Gemma evaluates a subset** in the ablation (compute honesty) — conclusions about the verifier are drawn from the stated sample, not the full test set.
4. **Video-level evaluation** follows from UCF-Crime's video-level labels; temporal localization is only qualitatively demonstrated (timeline plots), not scored against frame-level annotations.
5. **The verifier can be wrong**: a multimodal model may miss subtle incidents (R2 exists precisely so extreme detector evidence is never silently discarded) and inherits the biases of its pretraining. Human oversight remains in the loop by design (REVIEW lane).

In [ ]:
# =================== HACKATHON RESULTS SUMMARY (actual measured values only) ===============
def _g(path, default="not measured"):
    cur = RESULTS
    for k in path.split("."):
        if not isinstance(cur, dict) or k not in cur:
            return default
        cur = cur[k]
    return cur

_tm = RESULTS.get("test_metrics", {})
_roc = next((v for k, v in _tm.items() if "ROC-AUC" in k), "not measured")
_pr = next((v for k, v in _tm.items() if "PR-AUC" in k), "not measured")
_case = next((c for c in RESULTS.get("demo_cases", []) if c.get("decision") == "ESCALATE"), None)

md = f"""### Hackathon Results Summary — every number below was measured in this notebook run

| Item | Value |
|---|---|
| **Test binary ROC-AUC** (video-level, one-shot eval) | **{_roc}** |
| Test PR-AUC (average precision) | {_pr} |
| Operating threshold (frozen on validation) | {_g('alarm_threshold')} — rule: {_g('threshold_rule')} |
| Backbone (validation A/B winner) | {BEST_BACKBONE} |
| SafetyFormer | {CFG.n_layers}-layer transformer, d={CFG.d_model}, ~{sf_params/1e6:.1f}M params |
| MIL configuration | top-k mean, alpha={_g('mil_alpha')}, loss={_g('loss')} (validation-selected) |
| Gemma 4 model (exact) | {_g('gemma.model_id', 'NOT LOADED — fallback run')} |
| Quantization | {_g('gemma.quantization')} |
| Local inference | {_g('gemma.local_inference')} (weights on this machine, no inference API) |
| Gemma latency (n={_g('gemma_latency.n_calls', 0)}) | mean {_g('gemma_latency.mean_s')}s · median {_g('gemma_latency.median_s')}s · p95 {_g('gemma_latency.p95_s')}s |
| Gemma activation rate | {_g('selective_compute.activation_rate')} ({_g('selective_compute.gemma_calls')} calls over {_g('selective_compute.subset_windows_analyzed')} windows) |
| VLM calls avoided vs per-window VLM | {_g('selective_compute.vlm_calls_avoided_vs_per_window')} ({_g('selective_compute.vlm_reduction_pct')}%) |
| False-positive dispatch reduction (subset, A→C) | {_g('fp_reduction_pct')}% (recall {_g('subset_recall_A')} → {_g('subset_recall_C')}) |
| Example verified incident | {(_case['sequence'] + ' — ' + _case['rationale']) if _case else 'see case studies in §17'} |
| Known limitation | Gemma verifier evaluated on a labeled test subset, not the full test set (see §20) |
"""
display(Markdown(md))
with open(Path(CFG.out_dir) / "results.json", "w") as f:
    json.dump(RESULTS, f, indent=2, default=str)
print("Full machine-readable results saved to results.json")

## 21 · Conclusion & Rubric Map

**AEGIS = continuous cheap triage + selective local multimodal verification.** SafetyFormer watches every window for milliseconds' worth of compute; suspicious moments are anonymized and cross-examined by an on-device Gemma 4 that must find visual evidence before anything is dispatched. False alarms die at the verifier; genuine incidents arrive with an evidence-backed, structured, encrypted report.

| Rubric criterion | Where this notebook earns it |
|---|---|
| **Gemma Integration (30)** | Gemma 4 (`gemma-4-E4B-it`, 4-bit NF4, **local**) is the forensic verifier that final decisions depend on (§14–16). It receives real anonymized frames, not telemetry summaries; the ablation (§18) quantifies its contribution; removing it reverts the system to an uninspectable score threshold. |
| **Innovation & Impact (30)** | *Selective multimodal reasoning*: measured Gemma activation rate & VLM-calls-avoided (§18); privacy-by-architecture (anonymize-then-inspect, all-local, encrypted dispatch, §7/§19); false-alarm reduction quantified on real held-out data (§18). |
| **Functionality (20)** | End-to-end demo from raw frames to encrypted dispatch on real test videos, with true-positive **and** false-positive case studies and Gemma's verbatim JSON (§17); leakage-asserted splits (§6); one-shot test evaluation (§12). |
| **Presentation (20)** | Single narrative from problem to product; every metric correctly named (ROC-AUC is never called accuracy); honest protocol notes and limitations (§18/§20); measured-values-only results summary (§19–20). |

*Built for Build with Gemma (Track 2: AI for Public Safety). Reproducible top-to-bottom with `Run All` on a Kaggle GPU session with the UCF-Crime frame dataset attached.*